In [120]:

# Set project root as the working directory

import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()

# If notebook is running from the notebooks folder,
# move the working directory to the project root.
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

os.chdir(PROJECT_ROOT)

# Make project root available for local imports
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)
print("Working directory:", Path.cwd())


Project root: c:\Users\Diana\Desktop\ai-projects\assignment1-brochures
Working directory: c:\Users\Diana\Desktop\ai-projects\assignment1-brochures


In [67]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
import sys
sys.path.append("..")
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

from urllib.parse import urljoin, urldefrag, urlsplit

In [36]:
# Initialize and constants

# Initialize and constants

load_dotenv(override=True)

api_key = os.getenv("GOOGLE_API_KEY")
MODEL = os.getenv("GEMINI_MODEL")

if not api_key:
    raise ValueError("GOOGLE_API_KEY was not found in .env")

if not MODEL:
    raise ValueError("GEMINI_MODEL was not found in .env")

openai = OpenAI(
    api_key=api_key,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

print(f"Gemini client initialized successfully. Model: {MODEL}")

Gemini client initialized successfully. Model: gemini-3.1-flash-lite


In [7]:
# for testing
links = fetch_website_links("https://edwarddonner.com")
links

['#wp--skip-link--target',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/2026/11/11/ai-sdlc-the-ai-native-software-engineer-in-the-agentic-era/',
 'https://edwarddonner.com/2026/11/11/ai-sdlc-the-ai-native-software-engineer-in-the-agentic-era/',
 'https://edwarddonner.com/2026/10/14/ai-fde-the-forward-deployed-engineer-agentic-ai-architect/',
 'https://edwarddonner.com/2026/10/14/ai-fde-the-forward

## First step: Select relevant company pages using Gemini

We collect candidate links from a company's website and use an LLM
to identify the pages most relevant to our brochures.

The system considers two audiences:
- Potential customers
- Potential job candidates

The model must return structured JSON containing the selected URLs.
Python will validate the response before collecting the pages.

In [8]:

link_system_prompt = """
You are selecting relevant pages from a company's website.

Your goal is to identify pages that provide useful information
for two audiences:
1. Potential customers.
2. Potential job candidates.

Selection rules:
- Select up to 3 additional pages.
- Select only URLs from the provided candidate list.
- Do not select the homepage.
- Do not select privacy policies or terms of service.
- Do not select email links or external websites.
- Prefer pages about products, services, company information,
  technology, careers, teams, and company culture.
- Choose pages that provide useful information for one or
  both target audiences.
- Prefer diversity of information. Avoid selecting multiple pages
  that substantially cover the same subject.

For every selected page, provide:
- type: A short description of the page category.
- url: The complete selected URL.
- reason: A clear explanation of why the page is relevant
  to potential customers and/or job candidates.

The reason must explain the actual value of the page,
not simply repeat its title.

Return ONLY a JSON object using this structure:

{
    "links": [
        {
            "type": "products",
            "url": "https://company.example/products",
            "reason": "Explains the company's products and helps potential customers understand its solutions."
        }
    ]
}

All three fields must be non-empty strings.

If no suitable pages exist, return:
{
    "links": []
}
"""


In [9]:

def normalize_candidate_links(base_url, raw_links):
    base_parts = urlsplit(base_url)

    if base_parts.scheme not in {"http", "https"} or not base_parts.hostname:
        raise ValueError("Invalid company website URL")

    # Treat www and non-www as the same website
    base_host = base_parts.hostname.lower().removeprefix("www.")

    normalized_links = []
    seen = set()

    for link in raw_links:
        if not isinstance(link, str) or not link.strip():
            continue

        # Resolve relative URLs
        absolute_url = urljoin(base_url, link.strip())

        # Remove URL fragments
        absolute_url = urldefrag(absolute_url).url

        parts = urlsplit(absolute_url)

        # Allow only HTTP and HTTPS
        if parts.scheme.lower() not in {"http", "https"}:
            continue

        if not parts.hostname:
            continue

        # Reject external websites
        candidate_host = parts.hostname.lower().removeprefix("www.")

        if candidate_host != base_host:
            continue

        # Create a duplicate key treating www and non-www as identical
        dedupe_key = (
            parts.scheme.lower(),
            candidate_host,
            parts.port,
            parts.path or "/",
            parts.query
        )

        if dedupe_key in seen:
            continue

        seen.add(dedupe_key)
        normalized_links.append(absolute_url)

    return normalized_links


def get_links_user_prompt(url):
    global normalized_candidate_links

    # Collect raw website links
    raw_links = fetch_website_links(url)

    # Normalize and filter candidate links
    normalized_candidate_links = normalize_candidate_links(
        url,
        raw_links
    )

    # Build the prompt for Gemini
    user_prompt = f"""
Here are the candidate links collected from the company website:
{url}

Select relevant pages for potential customers and job candidates.

Use only the following normalized candidate URLs:

"""

    user_prompt += "\n".join(normalized_candidate_links)

    return user_prompt


In [10]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here are the candidate links collected from the company website:
https://edwarddonner.com

Select relevant pages for potential customers and job candidates.

Use only the following normalized candidate URLs:

https://edwarddonner.com
https://edwarddonner.com/avatar/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/2026/11/11/ai-sdlc-the-ai-native-software-engineer-in-the-agentic-era/
https://edwarddonner.com/2026/10/14/ai-fde-the-forward-deployed-engineer-agentic-ai-architect/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/


In [ ]:
# this is a new addition tat was added later for testing section
def validate_link_selection(raw_text, base_url, candidate_links):
    """
    Validate a model-generated link selection without API calls.

    Returns the validated selection or raises ValueError.
    Rejects the entire selection if any item is invalid.
    """
    if not raw_text:
        raise ValueError("Empty model response")

    try:
        selected = json.loads(raw_text)
    except json.JSONDecodeError as error:
        raise ValueError(f"Malformed JSON: {error}") from error

    if not isinstance(selected, dict):
        raise ValueError("Response must be a JSON object")

    links = selected.get("links")

    if not isinstance(links, list):
        raise ValueError("Response must contain a links list")

    if len(links) > 3:
        raise ValueError("More than 3 additional pages selected")

    seen_urls = set()
    validated_links = []

    for index, link in enumerate(links):
        if not isinstance(link, dict):
            raise ValueError(f"Link {index} must be an object")

        for field in ("type", "url", "reason"):
            if not isinstance(link.get(field), str) or not link[field].strip():
                raise ValueError(
                    f"Link {index} has a missing or empty '{field}' field"
                )

        selected_url = link["url"].strip()

        normalized_url = urldefrag(
            urljoin(base_url, selected_url)
        ).url

        if normalized_url not in candidate_links:
            raise ValueError(
                f"URL not in candidate list: {selected_url}"
            )

        if normalized_url in seen_urls:
            raise ValueError(
                f"Duplicate URL: {normalized_url}"
            )

        # Additional pages must not be the homepage.
        if normalized_url.rstrip("/") == base_url.rstrip("/"):
            raise ValueError("Homepage cannot be selected as an additional page")

        # Reject privacy and terms pages.
        path = urlsplit(normalized_url).path.lower()

        forbidden_segments = {
            "privacy", "privacy-policy",
            "terms", "terms-of-use",
            "terms-and-conditions"
        }

        if any(segment in forbidden_segments for segment in path.split("/")):
            raise ValueError(
                f"Forbidden page selected: {normalized_url}"
            )

        seen_urls.add(normalized_url)

        validated_links.append({
            "type": link["type"].strip(),
            "url": normalized_url,
            "reason": link["reason"].strip()
        })

    return {"links": validated_links}


In [44]:

def select_relevant_links(url):
    # Ask Gemini to select relevant pages
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )

    raw_text = response.choices[0].message.content

    # Keep the raw response for auditing
    raw_selection = raw_text

    try:
        if not raw_text:
            raise ValueError("Gemini returned an empty response")

        selected = json.loads(raw_text)

        # Validate the main structure
        if not isinstance(selected, dict):
            raise ValueError("Response must be a JSON object")

        if "links" not in selected or not isinstance(selected["links"], list):
            raise ValueError("Response must contain a links list")

        links = selected["links"]

        if len(links) > 3:
            raise ValueError("Gemini selected more than 3 additional pages")

        seen_urls = set()

        for index, link in enumerate(links):
            if not isinstance(link, dict):
                raise ValueError(f"Link {index} must be an object")

            # Validate required fields
            for field in ["type", "url", "reason"]:
                if not isinstance(link.get(field), str) or not link[field].strip():
                    raise ValueError(
                        f"Link {index} has a missing or empty '{field}' field"
                    )

            selected_url = link["url"].strip()

            # Normalize the selected URL before checking it
            normalized_url = urldefrag(
                urljoin(url, selected_url)
            ).url

            # Ensure the selected URL was provided to Gemini
            if normalized_url not in normalized_candidate_links:
                raise ValueError(
                    f"Selected URL was not in the candidate list: {selected_url}"
                )

            # Reject duplicate selections
            if normalized_url in seen_urls:
                raise ValueError(
                    f"Duplicate selected URL: {normalized_url}"
                )

            seen_urls.add(normalized_url)

            # Store the normalized URL
            link["url"] = normalized_url

        print("Gemini selection validated successfully.")
        return selected

    except (json.JSONDecodeError, ValueError) as error:
        # Save rejected response and reason
        os.makedirs("logs", exist_ok=True)

        rejection = {
            "raw_response": raw_selection,
            "rejection_reason": str(error)
        }

        with open("logs/link_selection_rejection.json", "w", encoding="utf-8") as file:
            json.dump(rejection, file, indent=4, ensure_ascii=False)

        raise ValueError(
            f"Gemini link selection rejected: {error}"
        ) from error


In [12]:
select_relevant_links("https://edwarddonner.com")

Gemini selection validated successfully.


{'links': [{'type': 'Company Information',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/',
   'reason': 'Provides essential background on the founder and the company, helping potential customers understand the business mission and candidates gauge company culture.'},
  {'type': 'Technical Expertise',
   'url': 'https://edwarddonner.com/proficient/',
   'reason': "Showcases the technical stack and capabilities, allowing customers to assess the company's skill set and candidates to determine if their expertise aligns with the firm."},
  {'type': 'Thought Leadership',
   'url': 'https://edwarddonner.com/2026/11/11/ai-sdlc-the-ai-native-software-engineer-in-the-agentic-era/',
   'reason': "Demonstrates the company's forward-thinking approach to AI development, attracting customers looking for modern solutions and candidates interested in working with cutting-edge methodologies."}]}

next part: collecting and storing the actual website pages

In [13]:

from datetime import datetime, timezone


def fetch_company_sources(company_id, url, selected):
    """
    Fetch the homepage and selected pages.
    Save each page as an identifiable source record.
    """

    # Company A gets A1-A4, Company B gets B1-B4
    source_prefix = company_id.upper()

    pages = [
        {
            "url": url,
            "type": "homepage"
        }
    ]

    # Add the selected pages in their original selection order
    for link in selected["links"]:
        pages.append({
            "url": link["url"],
            "type": link["type"]
        })

    sources = []

    for index, page_info in enumerate(pages, start=1):

        source_id = f"{source_prefix}{index}"
        page_url = page_info["url"]

        print(f"Fetching {source_id}: {page_url}")

        source = {
            "source_id": source_id,
            "company_id": source_prefix,
            "url": page_url,
            "title": "",
            "retrieved_at": datetime.now(timezone.utc).isoformat(),
            "text": "",
            "status": "failed"
        }

        try:
            content = fetch_website_contents(page_url)

            if content:
                title, separator, text = content.partition("\n\n")

                source["title"] = title.strip()
                source["text"] = text.strip() if separator else ""

                if source["text"]:
                    source["status"] = "ok"
                else:
                    source["status"] = "empty"
            else:
                source["status"] = "empty"

        except Exception as error:
            print(f"Failed to fetch {source_id}: {error}")
            source["status"] = "failed"

        sources.append(source)

    # Save the source collection locally
    os.makedirs("data/sources", exist_ok=True)

    output_path = f"data/sources/{source_prefix}_sources.json"

    with open(output_path, "w", encoding="utf-8") as file:
        json.dump(sources, file, indent=4, ensure_ascii=False)

    print(f"\nSaved {len(sources)} sources to {output_path}")

    return sources


In [14]:
selected_links = select_relevant_links(
    "https://edwarddonner.com"
)

company_a_sources = fetch_company_sources(
    company_id="A",
    url="https://edwarddonner.com",
    selected=selected_links
)

company_a_sources

Gemini selection validated successfully.
Fetching A1: https://edwarddonner.com
Fetching A2: https://edwarddonner.com/about-me-and-about-nebula/
Fetching A3: https://edwarddonner.com/2026/11/11/ai-sdlc-the-ai-native-software-engineer-in-the-agentic-era/
Failed to fetch A3: 429 Client Error: Too Many Requests for url: https://edwarddonner.com/2026/11/11/ai-sdlc-the-ai-native-software-engineer-in-the-agentic-era/
Fetching A4: https://edwarddonner.com/proficient/

Saved 4 sources to data/sources/A_sources.json


[{'source_id': 'A1',
  'company_id': 'A',
  'url': 'https://edwarddonner.com',
  'title': 'Home - Edward Donner',
  'retrieved_at': '2026-11-18T16:37:35.032623+00:00',
  'text': 'Skip to content\nAvatar\nCurriculum\nProficiency\nC4\nOutsmart\nAn arena that pits LLMs against each other in a battle of diplomacy and deviousness\nAbout\nPosts\nWell, hi there.\nI’m Ed. I like writing code and experimenting with LLMs, and hopefully you’re here because you do too. I also enjoy amateur electronic music production (\nvery\namateur) and losing myself in\nHacker News\n, nodding my head sagely to things I only half understand.\nI’m the co-founder and CTO of AI startup\nNebula.io\n. I was previously founder and CEO of AI startup untapt,\nacquired in 2021\n, and a Managing Director at JPMorgan.\nI will happily drone on for hours about LLMs to anyone in my vicinity. My friends got fed up with my impromptu lectures, and convinced me to make some Udemy courses. To my total joy (and shock) they’ve becom

In [15]:
# Context configuration

CONTEXT_TEXT_BUDGET = 12_000  # Characters, not tokens

In [100]:

def build_context(sources, text_budget=CONTEXT_TEXT_BUDGET, output_name="company"):
    """
    Build a context from saved source records.

    The complete context, including source metadata,
    must stay within the character budget.
    Only successfully collected sources are used.
    """

    valid_sources = [
        source for source in sources
        if source["status"] == "ok" and source["text"].strip()
    ]

    omitted_sources = [
        source for source in sources
        if source not in valid_sources
    ]

    context_parts = []
    included = []
    truncated = []

    # Calculate the metadata overhead first
    metadata_parts = []

    for source in valid_sources:
        metadata = (
            f"## Source {source['source_id']}\n"
            f"Title: {source['title']}\n"
            f"URL: {source['url']}\n"
            f"Content:\n"
        )
        metadata_parts.append(metadata)

    metadata_cost = sum(len(part) for part in metadata_parts)

    # Account for separators between sources
    separator_cost = max(0, len(valid_sources) - 1) * 2

    available_text_budget = max(
        0,
        text_budget - metadata_cost - separator_cost
    )

    # Divide the remaining budget equally
    source_limit = (
        available_text_budget // len(valid_sources)
        if valid_sources else 0
    )

    for source, metadata in zip(valid_sources, metadata_parts):

        original_text = source["text"]
        selected_text = original_text[:source_limit]

        context_parts.append(metadata + selected_text)

        included.append(source["source_id"])

        if len(selected_text) < len(original_text):
            truncated.append({
                "source_id": source["source_id"],
                "original_characters": len(original_text),
                "included_characters": len(selected_text),
                "reason": "Complete context budget allocation"
            })

    context = "\n\n".join(context_parts)

    # Record actual context length
    manifest = {
        "output_name": output_name,
        "text_budget": text_budget,
        "characters_used": len(context),
        "included_sources": included,
        "truncated_sources": truncated,
        "omitted_sources": [
            {
                "source_id": source["source_id"],
                "reason": f"Source status: {source['status']}"
            }
            for source in omitted_sources
        ]
    }

    # Save context and manifest
    os.makedirs("data/contexts", exist_ok=True)

    context_path = f"data/contexts/{output_name}_context.txt"
    manifest_path = f"data/contexts/{output_name}_manifest.json"

    with open(context_path, "w", encoding="utf-8") as file:
        file.write(context)

    with open(manifest_path, "w", encoding="utf-8") as file:
        json.dump(manifest, file, indent=4, ensure_ascii=False)

    print(f"Context created: {output_name}")
    print(
        f"Characters used: "
        f"{manifest['characters_used']}/{text_budget}"
    )
    print(f"Included sources: {included}")
    print(f"Truncated sources: {len(truncated)}")
    print(f"Omitted sources: {len(omitted_sources)}")

    return context, manifest


In [ ]:
# testing with edwarddonner.com sources
with open("data/sources/A_sources.json", "r", encoding="utf-8") as file:
    company_a_sources = json.load(file)

print(f"Loaded {len(company_a_sources)} sources")

Loaded 4 sources


In [18]:
company_a_context, company_a_manifest = build_context(
    company_a_sources,
    output_name="company_A"
)

Context created: company_A
Characters used: 5363/12000
Included sources: ['A1', 'A2', 'A4']
Truncated sources: 0
Omitted sources: 1


In [19]:
# Brochure prompt builder
def build_brochure_prompt(company_name, audience, context):

    if audience == "customers":
        audience_instructions = """
        Write for potential customers.

        Focus on:
        - The problem the company aims to solve.
        - Its products and services.
        - Practical use cases.
        - Target audiences.
        - What potential customers should clarify before contacting the company.
        """

    elif audience == "candidates":
        audience_instructions = """
        Write for potential job candidates.

        Focus on:
        - The company's business activities.
        - Published roles and teams.
        - Work environment and company culture,
          only when supported by the sources.
        - The application process, if described.
        - What candidates should clarify before applying.
        """

    else:
        raise ValueError("Audience must be 'customers' or 'candidates'")

    prompt = f"""
You are writing a company brochure in Hebrew.

COMPANY:
{company_name}

TARGET AUDIENCE:
{audience}

INSTRUCTIONS:
{audience_instructions}

GENERAL REQUIREMENTS:

1. Write between 200 and 300 words.
   Count the words using Python's split() method.
   The source list at the end is excluded from the count.

2. Use clear Markdown headings and an organized structure.

3. Support every substantial factual claim with a source
   reference in this format: [A1], [A2], etc.

4. Only reference source IDs included in the supplied context.

5. Do not invent customers, prices, job openings,
   employment conditions or company promises.

6. Attribute marketing claims to the company that published them.

7. If important information is unavailable, write:
   "לא נמצא במקורות שנמסרו למודל".

8. Keep company facts consistent across audiences.
   Adapt the structure and emphasis to the reader,
   not the underlying facts.

9. Finish with a section titled "מקורות" containing
   the source IDs and URLs used.

SOURCE MATERIAL:
----------------
{context}
----------------

Write the complete brochure in Hebrew.
"""

    return prompt


In [21]:

from pathlib import Path
from uuid import uuid4
import re

def generate_brochure(company_name, company_id, audience, context):

    # Build the prompt using our existing function
    prompt = build_brochure_prompt(
        company_name=company_name,
        audience=audience,
        context=context
    )

    # Create a unique ID for this generation
    run_id = str(uuid4())

    # Save the exact prompt sent to Gemini
    output_dir = Path("outputs/brochures")
    prompt_dir = Path("outputs/prompts")

    output_dir.mkdir(parents=True, exist_ok=True)
    prompt_dir.mkdir(parents=True, exist_ok=True)

    (prompt_dir / f"{run_id}.txt").write_text(
        prompt, encoding="utf-8"
    )

    # Send the request to Gemini
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "user", "content": prompt}
        ]
    )

    brochure = response.choices[0].message.content or ""

    # Separate the brochure from the source list
    brochure_body = re.split(
        r"(?m)^#{1,6}\s*מקורות\s*$",
        brochure,
        maxsplit=1
    )[0]

    word_count = len(brochure_body.split())

    # Save the generated brochure
    output_path = output_dir / f"{company_id}_{audience}_{run_id}.md"

    output_path.write_text(
        brochure, encoding="utf-8"
    )

    print(f"Run ID: {run_id}")
    print(f"Saved to: {output_path}")
    print(f"Word count: {word_count}")

    if not 200 <= word_count <= 300:
        print("WARNING: Brochure is outside the required 200-300 words.")

    return brochure


In [ ]:
# testing customer brochure generation for company A
company_a_customer_brochure = generate_brochure(
    company_name="Edward Donner",
    company_id="A",
    audience="customers",
    context=company_a_context
)

print(company_a_customer_brochure)

Run ID: fbae19ea-644b-4199-9fa8-e88363f62814
Saved to: outputs\brochures\A_customers_fbae19ea-644b-4199-9fa8-e88363f62814.md
Word count: 251
### פתרונות AI מתקדמים עם אדוארד דונר

בעולם העבודה המודרני, אתגרים טכנולוגיים ושינויים בשוק דורשים מומחיות מעמיקה בתחומי הבינה המלאכותית. אדוארד דונר, מייסד שותף ו-CTO בחברת Nebula.io, רותם את ניסיונו העשיר בתחום ה-AI כדי לסייע לעסקים ולאנשי מקצוע לנווט בעידן ה-Agentic AI [A1, A2].

#### הבעיה שאנו פותרים
בתחום הגיוס והשמת כוח אדם, חברות רבות מתמודדות עם קשיים בזיהוי וניהול כישרונות, כאשר נתונים מצביעים על כך ש-77% מהעובדים אינם חשים השראה או מעורבות בעבודתם [A2]. אדוארד דונר פועל לפתרון בעיות מורכבות אלו באמצעות מודלים של בינה מלאכותית יוצרת (Generative AI) [A2].

#### שירותים ומוצרים
*   **טכנולוגיית Nebula.io:** מערכת המשתמשת במודל מוגן בפטנט כדי לבצע התאמה מדויקת בין מועמדים לתפקידים, ללא צורך במילות מפתח, ביעילות גבוהה מאי פעם [A2].
*   **הכשרה מקצועית:** דונר מציע תוכניות לימוד מקיפות בתחומי בניית AI, הנדסת AI ומנהיגות ב-AI, אשר זכו ללמעלה 

In [23]:

import re

def validate_brochure_citations(brochure, manifest):

    # IDs of sources included in the context
    allowed_ids = set(manifest["included_sources"])

    # Extract individual IDs, including from [A1, A2]
    citations = re.findall(r"\b[A-Z][1-4]\b", brochure)

    cited_ids = set(citations)

    invalid_ids = cited_ids - allowed_ids

    print("Allowed source IDs:", sorted(allowed_ids))
    print("Cited source IDs:", sorted(cited_ids))
    print("Invalid source IDs:", sorted(invalid_ids))

    return len(invalid_ids) == 0


In [24]:
is_valid = validate_brochure_citations(
    company_a_customer_brochure,
    company_a_manifest
)

print("Citation validation passed:", is_valid)

Allowed source IDs: ['A1', 'A2', 'A4']
Cited source IDs: ['A1', 'A2', 'A4']
Invalid source IDs: []
Citation validation passed: True


In [25]:
# testing candidate brochure for company A 

company_a_candidate_brochure = generate_brochure(
    company_name="Edward Donner",
    company_id="A",
    audience="candidates",
    context=company_a_context
)

print(company_a_candidate_brochure)


Run ID: b6b15ced-3bd4-4894-b886-8708a7b6a76f
Saved to: outputs\brochures\A_candidates_b6b15ced-3bd4-4894-b886-8708a7b6a76f.md
Word count: 251
### הצטרפו לעשייה ב-Nebula.io

אדוארד דונר, מייסד שותף ו-CTO בחברת הסטארט-אפ בתחום הבינה המלאכותית **Nebula.io**, מזמין מועמדים לבחון הזדמנויות בעולם שבו טכנולוגיית ה-AI משכתבת את גבולות האפשר [A1, A2].

#### מה אנחנו עושים?
ב-Nebula.io, המשימה שלנו היא לסייע למגייסים לאתר, להבין, לערב ולנהל כישרונות באמצעות שימוש בבינה מלאכותית יוצרת (Generative AI) ושיטות למידת מכונה מתקדמות [A2]. החברה פיתחה מודל פטנטי המבצע התאמה בין אנשים לתפקידים בדיוק ובמהירות גבוהים, ללא צורך במילות מפתח [A2]. המטרה ארוכת הטווח של החברה היא לסייע לאנשים לממש את הפוטנציאל שלהם ולמצוא תפקידים שבהם ירגישו הגשמה והצלחה [A2].

#### צוותים ותפקידים פתוחים
לא נמצא במקורות שנמסרו למודל מידע לגבי משרות פתוחות ספציפיות או פירוט מבני של צוותים בחברה. עם זאת, אדוארד דונר מדגיש את חשיבותן של מיומנויות בבניית AI, הנדסת AI ומנהיגות בתחום ה-AI, ומפעיל מדריך ייעודי למהנדסים מיומנים שסיימו

In [26]:

candidate_is_valid = validate_brochure_citations(
    company_a_candidate_brochure,
    company_a_manifest
)

print("Candidate brochure citation validation:", candidate_is_valid)


Allowed source IDs: ['A1', 'A2', 'A4']
Cited source IDs: ['A1', 'A2', 'A4']
Invalid source IDs: []
Candidate brochure citation validation: True


both broshures worked - testing is done, now moving on to real companies brochure

In [27]:
COMPANIES = {
    "A": {
        "name": "NVIDIA",
        "url": "https://www.nvidia.com/"
    },
    "B": {
        "name": "AMD",
        "url": "https://www.amd.com/en.html"
    }
}

In [64]:
# Collect NVIDIA website sources

nvidia_url = COMPANIES["A"]["url"]

nvidia_selected_links = select_relevant_links(nvidia_url)

nvidia_sources = fetch_company_sources(
    company_id="A",
    url=nvidia_url,
    selected=nvidia_selected_links
)

Gemini selection validated successfully.
Fetching A1: https://www.nvidia.com/
Fetching A2: https://www.nvidia.com/en-eu/about-nvidia/careers/
Fetching A3: https://www.nvidia.com/en-eu/about-nvidia/
Fetching A4: https://www.nvidia.com/en-eu/software/

Saved 4 sources to data/sources/A_sources.json


In [65]:
# Collect AMD website sources

amd_url = COMPANIES["B"]["url"]

amd_selected_links = select_relevant_links(amd_url)

amd_sources = fetch_company_sources(
    company_id="B",
    url=amd_url,
    selected=amd_selected_links
)

Gemini selection validated successfully.
Fetching B1: https://www.amd.com/en.html
Fetching B2: https://www.amd.com/en/corporate/careers.html
Fetching B3: https://www.amd.com/en/products/processors/server/epyc.html
Fetching B4: https://www.amd.com/en/corporate.html

Saved 4 sources to data/sources/B_sources.json


In [72]:
# Build NVIDIA context
nvidia_context, nvidia_manifest = build_context(
    nvidia_sources,
    output_name="NVIDIA"
)

# Build AMD context
amd_context, amd_manifest = build_context(
    amd_sources,
    output_name="AMD"
)

Context created: NVIDIA
Characters used: 7801/12000
Included sources: ['A1', 'A2', 'A3', 'A4']
Truncated sources: 0
Omitted sources: 0
Context created: AMD
Characters used: 7882/12000
Included sources: ['B1', 'B2', 'B3', 'B4']
Truncated sources: 0
Omitted sources: 0


In [75]:
# Create the brochure generation function

def generate_brochure(company_name, context, audience):
    """
    Generate a company brochure for a specific audience.
    """

    print(f"Generating {company_name} brochure for {audience}...")

    # Build the complete prompt
    prompt = build_brochure_prompt(
        company_name,
        audience,
        context
    )

    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    return response.choices[0].message.content


In [76]:
# Generate NVIDIA customer brochure

nvidia_customers_brochure = generate_brochure(
    company_name="NVIDIA",
    context=nvidia_context,
    audience="customers"
)

print(nvidia_customers_brochure)

Generating NVIDIA brochure for customers...
### מובילה עולמית במחשוב בינה מלאכותית: NVIDIA

חברת NVIDIA מובילה את מהפכת המחשוב המואץ במטרה להתמודד עם אתגרים טכנולוגיים מורכבים שאף גורם אחר לא הצליח לפתור [A3]. החברה פועלת כ"מנוע של הבינה המלאכותית" ומספקת את השבבים, המערכות והתוכנות המתקדמים ביותר עבור "מפעלי הבינה המלאכותית" של העתיד [A3].

#### הפתרונות והמוצרים שלנו
NVIDIA מפתחת טכנולוגיות המשנות תעשיות שלמות, תוך התמקדות במספר תחומים מרכזיים:
*   **מחשוב בעל ביצועים גבוהים (HPC):** פלטפורמות כדוגמת Vera מאיצות תגליות מדעיות וסימולציות מורכבות [A1].
*   **בינה מלאכותית ו-NIM:** שירותי מיקרו (Microservices) ומסגרות עבודה כגון NeMo מאפשרים לארגונים לבנות ולפרוס פתרונות בינה מלאכותית בהתאמה אישית [A4].
*   **דיגיטציה תעשייתית:** פלטפורמת Omniverse מאפשרת יצירת תאומים דיגיטליים (Digital Twins) וחיבור בין העולם הפיזי לדיגיטלי [A3].
*   **תחבורה אוטונומית:** פתרונות NVIDIA DRIVE מעניקים לרכבים את היכולת לראות, ללמוד ולנווט במגוון רחב של תרחישים [A2, A4].
*   **גרפיקה:** טכנולוגיית NVIDIA 

In [77]:
# validation of citations for NVIDIA customer brochure

import re

def validate_brochure(brochure, allowed_source_ids):

    # Separate the brochure from its source list
    parts = re.split(
        r"(?im)^#{1,6}\s*מקורות\s*$",
        brochure,
        maxsplit=1
    )

    main_content = parts[0]

    # Count words excluding the source list
    word_count = len(main_content.split())

    # Find all source citations
    cited_ids = set(
        re.findall(r"\[(A\d+|B\d+)\]", main_content)
    )

    # Check for invalid source IDs
    invalid_ids = cited_ids - set(allowed_source_ids)

    print("Word count:", word_count)
    print("Word count valid:", 200 <= word_count <= 300)

    print("\nAllowed source IDs:", allowed_source_ids)
    print("Cited source IDs:", sorted(cited_ids))
    print("Invalid source IDs:", sorted(invalid_ids))

    print(
        "\nCitation ID validation:",
        len(invalid_ids) == 0
    )

    return {
        "word_count": word_count,
        "word_count_valid": 200 <= word_count <= 300,
        "cited_ids": cited_ids,
        "invalid_ids": invalid_ids
    }


In [78]:
nvidia_customer_validation = validate_brochure(
    nvidia_customers_brochure,
    [source["source_id"] for source in nvidia_sources]
)

Word count: 278
Word count valid: True

Allowed source IDs: ['A1', 'A2', 'A3', 'A4']
Cited source IDs: ['A1', 'A2', 'A3', 'A4']
Invalid source IDs: []

Citation ID validation: True


In [79]:
for source in nvidia_sources:
    print(f"\n{'=' * 60}")
    print(source["source_id"], source["title"])
    print(source["url"])
    print(source["text"][:1500])


A1 World Leader in Artificial Intelligence Computing | NVIDIA
https://www.nvidia.com/
Artificial Intelligence Computing Leadership from NVIDIA
HPC
NVIDIA Introduces Nemotron for Science to Accelerate Agentic Discovery
New open dataset and post-training recipes will help researchers adapt Nemotron 3.5 Super VL for agents in life sciences and chemistry.
Read Press Release
Data Center
NVIDIA Expands Vera Platform to Accelerate Scientific Discovery
NVIDIA’s Vera CPU rack-scale platform pairs its first custom data center CPU with top HPC networking, turbocharging scientific modeling, simulation, and agentic science.
Read Announcement
Forschungszentrum Jülich / Sascha Kreklau
HPC
NVIDIA, IBM, Fermilab Announce Codesign Quantum Computing Platform
New TCBP-JOBS decoder in CUDA-Q™ QEC delivers up to a 13,000x speedup for quantum error-correction research.
Read Announcement
Networking
Lambda Boosts AI Factory Resiliency, Efficiency With NVIDIA Photonics
The world’s first large-scale co-packaged

In [80]:

from pathlib import Path
from uuid import uuid4

def save_brochure(company_id, audience, brochure):

    run_id = str(uuid4())

    output_dir = Path("outputs/brochures")
    output_dir.mkdir(parents=True, exist_ok=True)

    filename = f"{company_id}_{audience}_{run_id}.md"
    output_path = output_dir / filename

    output_path.write_text(
        brochure,
        encoding="utf-8"
    )

    print("Brochure saved successfully!")
    print("Run ID:", run_id)
    print("Saved to:", output_path)

    return output_path


In [81]:
# save the brochure we already generated
nvidia_customers_path = save_brochure(
    company_id="A",
    audience="customers",
    brochure=nvidia_customers_brochure
)

Brochure saved successfully!
Run ID: c332881b-5997-4c41-8a1e-1175512a94a6
Saved to: outputs\brochures\A_customers_c332881b-5997-4c41-8a1e-1175512a94a6.md


In [82]:
# Generate NVIDIA candidate brochure

nvidia_candidates_brochure = generate_brochure(
    company_name="NVIDIA",
    context=nvidia_context,
    audience="candidates"
)

print(nvidia_candidates_brochure)

Generating NVIDIA brochure for candidates...
### קריירה ב-NVIDIA: הצטרפו להובלת מהפכת הבינה המלאכותית

NVIDIA מובילה את תחום המחשוב המואץ ובינה מלאכותית, תוך התמודדות עם האתגרים המורכבים בעולם [A1, A3]. החברה מפתחת את השבבים, המערכות והתוכנות המתקדמים ביותר עבור "מפעלי הבינה המלאכותית" של העתיד, ופועלת בתחומים משני תעשיות כגון בריאות, רכב אוטונומי, רובוטיקה ותאומים דיגיטליים [A2, A3].

#### צוותים ותחומי עיסוק
אנו מחפשים אנשי מקצוע נלהבים להצטרף לצוותים המגוונים שלנו, ביניהם:
*   **בינה מלאכותית (AI):** פיתוח טכנולוגיות המאפשרות לעסקים לנתח נתונים ולקבל החלטות חכמות ומהירות יותר [A2].
*   **רכב (Auto):** פיתוח יכולות של ראייה, למידה וניווט עבור כלי רכב אוטונומיים [A2].
*   **חומרה:** צוותי הנדסה העובדים על קווי מוצרים החל מגרפיקה לצרכנים ועד לתשתית עבור בינה מלאכותית ומחשוב מואץ [A2].

#### סביבת עבודה ותרבות ארגונית
NVIDIA מתוארת כמקום עבודה ייחודי ("Like No Place You’ve Ever Worked"). תרבות החברה מבוססת על שיתוף פעולה הדוק, הצלחה משותפת, והכלה של אינדיבידואליות. אנו מאמינים כי "הפרוי

In [85]:
# save the NVIDIA candidate brochure
nvidia_candidates_path = save_brochure(
    company_id="A",
    audience="candidates",
    brochure=nvidia_candidates_brochure
)

Brochure saved successfully!
Run ID: b9839cc2-660b-4f76-ba9f-48aeff5992ba
Saved to: outputs\brochures\A_candidates_b9839cc2-660b-4f76-ba9f-48aeff5992ba.md


In [86]:
# Generate AMD customer brochure

amd_customers_brochure = generate_brochure(
    company_name="AMD",
    context=amd_context,
    audience="customers"
)

print(amd_customers_brochure)

Generating AMD brochure for customers...
# פתרונות מחשוב מתקדמים מבית AMD

בעולם שבו הבינה המלאכותית (AI) מעצבת מחדש כל היבט עסקי, ארגונים ניצבים בפני האתגר להפוך שאיפות לתוצאות ממשיות. חברת AMD מספקת את התשתית הטכנולוגית המהימנה והגמישה הנדרשת כדי לצלוח את המעבר הזה [B1].

### מה אנו מציעים?
AMD היא מובילה עולמית בתחומי המחשוב עתיר הביצועים והמחשוב האדפטיבי [B4]. אנו החברה היחידה המציעה פורטפוליו רחב הכולל פתרונות CPU, GPU וטכנולוגיות מחשוב אדפטיבי, המאפשרים הטמעת פתרונות AI מותאמים אישית לצרכי העסק שלכם [B1].

המוצרים והשירותים שלנו כוללים:
*   **מעבדי AMD EPYC™:** דורות חדשים של מעבדי שרתים המציעים ביצועים יוצאי דופן, יעילות אנרגטית וחיסכון בעלויות עבור מרכזי נתונים, עננים וסביבות עבודה עתירות דרישות [B3].
*   **פתרונות AI מקצה לקצה:** ממרכזי נתונים ועד לקצה (Edge), אנו מספקים ארכיטקטורה המבוססת על גישת "מערכת אקולוגית פתוחה", המבטיחה גמישות והגנה על השקעות הארגון [B1].

### למי הפתרונות שלנו מתאימים?
הטכנולוגיות שלנו משרתות מגוון רחב של לקוחות, החל ממרכזי נתונים ארגוניים וספקי ענן,

In [89]:
amd_customers_path = save_brochure(
    company_id="B",
    audience="customers",
    brochure=amd_customers_brochure
)

Brochure saved successfully!
Run ID: 19b70913-5646-491e-9710-851aad2887f3
Saved to: outputs\brochures\B_customers_19b70913-5646-491e-9710-851aad2887f3.md


In [90]:
# Generate AMD candidate brochure

amd_candidates_brochure = generate_brochure(
    company_name="AMD",
    context=amd_context,
    audience="candidates"
)

print(amd_candidates_brochure)

Generating AMD brochure for candidates...
### בואו להוביל את העתיד הטכנולוגי ב-AMD

האם אתם מחפשים להשפיע על עתיד המחשוב וה-AI? ב-AMD, אנו מזמינים אתכם להצטרף לחברה המובילה בתחומי מחשוב בעל ביצועים גבוהים ומחשוב אדפטיבי [B4]. אנו פועלים לפיתוח טכנולוגיות מתקדמות המניעות את עולמות הדאטה סנטר, הענן, הגיימינג והבינה המלאכותית [B1, B4].

#### תחומי פעילות וצוותים
ב-AMD תמצאו סביבה שבה כל תפקיד תורם לקידום הטכנולוגיה העולמית. החברה מציעה הזדמנויות בתחומים מגוונים, כולל:
*   **הנדסה:** תכנון הדור הבא של מעבדים (כגון סדרת EPYC) ופריצות דרך בתחום ה-AI [B2, B3].
*   **שיווק ומכירות:** יצירת אסטרטגיות כניסה לשוק [B2].
*   **תפקידים תאגידיים:** תמיכה בתפעול הגלובלי של החברה [B2].
*   **תוכניות סטודנטים:** הזדמנויות התמחות ופיתוח קריירה מוקדמת בלב האקו-סיסטם של ה-AI [B2].

#### סביבת עבודה ותרבות ארגונית
AMD מגדירה את התרבות שלה כקהילה שבה "יחד אנחנו מתקדמים" (Together we advance) [B2]. החברה מדגישה עבודה בצוות המבוססת על אינקלוסיביות, תמיכה וחיבור בין עובדים [B2, B4]. העובדים נהנים מיתרונות כמו ג

In [93]:
amd_candidates_path = save_brochure(
    company_id="B",
    audience="candidates",
    brochure=amd_candidates_brochure
)

Brochure saved successfully!
Run ID: d280c476-d526-4c2d-9513-73a85eca2dac
Saved to: outputs\brochures\B_candidates_d280c476-d526-4c2d-9513-73a85eca2dac.md


In [94]:
# final workflow

# Finalize the four existing brochures
# No text modifications and no Gemini API calls.

from pathlib import Path

brochures_to_finalize = [
    {
        "name": "NVIDIA Customers",
        "company_id": "A",
        "audience": "customers",
        "brochure": nvidia_customers_brochure,
        "sources": nvidia_sources
    },
    {
        "name": "NVIDIA Candidates",
        "company_id": "A",
        "audience": "candidates",
        "brochure": nvidia_candidates_brochure,
        "sources": nvidia_sources
    },
    {
        "name": "AMD Customers",
        "company_id": "B",
        "audience": "customers",
        "brochure": amd_customers_brochure,
        "sources": amd_sources
    },
    {
        "name": "AMD Candidates",
        "company_id": "B",
        "audience": "candidates",
        "brochure": amd_candidates_brochure,
        "sources": amd_sources
    }
]

final_report = []

for item in brochures_to_finalize:

    print(f"\n{'=' * 50}")
    print(f"Processing: {item['name']}")
    print("=" * 50)

    allowed_ids = [
        source["source_id"]
        for source in item["sources"]
    ]

    validation = validate_brochure(
        item["brochure"],
        allowed_ids
    )

    passed = (
        validation["word_count_valid"]
        and len(validation["invalid_ids"]) == 0
        and bool(item["brochure"].strip())
    )

    saved_path = None

    if passed:
        saved_path = save_brochure(
            company_id=item["company_id"],
            audience=item["audience"],
            brochure=item["brochure"]
        )

        print("Status: PASSED")
    else:
        print("Status: FAILED")

    final_report.append({
        "brochure": item["name"],
        "word_count": validation["word_count"],
        "citations_valid": len(validation["invalid_ids"]) == 0,
        "status": "PASSED" if passed else "FAILED",
        "saved_path": str(saved_path) if saved_path else None
    })


# Final report
print("\n========== FINAL REPORT ==========")

for result in final_report:
    print(f"\n{result['brochure']}")
    print("Word count:", result["word_count"])
    print("Citations valid:", result["citations_valid"])
    print("Status:", result["status"])
    print("Saved to:", result["saved_path"])

print("\nFinalization completed.")



Processing: NVIDIA Customers
Word count: 278
Word count valid: True

Allowed source IDs: ['A1', 'A2', 'A3', 'A4']
Cited source IDs: ['A1', 'A2', 'A3', 'A4']
Invalid source IDs: []

Citation ID validation: True
Brochure saved successfully!
Run ID: 27d2a67e-38f3-4018-a195-64ae68e8a2e2
Saved to: outputs\brochures\A_customers_27d2a67e-38f3-4018-a195-64ae68e8a2e2.md
Status: PASSED

Processing: NVIDIA Candidates
Word count: 246
Word count valid: True

Allowed source IDs: ['A1', 'A2', 'A3', 'A4']
Cited source IDs: ['A1', 'A2', 'A3', 'A4']
Invalid source IDs: []

Citation ID validation: True
Brochure saved successfully!
Run ID: 409e3b4f-8c07-4a0f-bc2b-502f5431da45
Saved to: outputs\brochures\A_candidates_409e3b4f-8c07-4a0f-bc2b-502f5431da45.md
Status: PASSED

Processing: AMD Customers
Word count: 260
Word count valid: True

Allowed source IDs: ['B1', 'B2', 'B3', 'B4']
Cited source IDs: ['B1', 'B3', 'B4']
Invalid source IDs: []

Citation ID validation: True
Brochure saved successfully!
Run ID:

## Company comparison

In [102]:
# Build a combined context for NVIDIA vs AMD comparison

combined_sources = nvidia_sources + amd_sources

comparison_context, comparison_manifest = build_context(
    combined_sources,
    text_budget=12000,
    output_name="NVIDIA_AMD_comparison"
)

assert len(comparison_context) <= 12000
assert len(comparison_manifest["included_sources"]) == 8
assert len(comparison_manifest["omitted_sources"]) == 0


Context created: NVIDIA_AMD_comparison
Characters used: 11993/12000
Included sources: ['A1', 'A2', 'A3', 'A4', 'B1', 'B2', 'B3', 'B4']
Truncated sources: 8
Omitted sources: 0


In [103]:

# Company comparison prompt builder

COMPARISON_DIMENSIONS = [
    "Products and services",
    "Target customers",
    "AI and computing capabilities",
    "Careers and work environment"
]


def build_comparison_prompt(context):

    dimensions = "\n".join(
        f"{i + 1}. {dimension}"
        for i, dimension in enumerate(COMPARISON_DIMENSIONS)
    )

    prompt = f"""
You are an expert business analyst.

Your task is to compare NVIDIA and AMD using only
the supplied company source material.

Write the comparison in Hebrew.

COMPANIES:
- NVIDIA
- AMD

COMPARISON DIMENSIONS:
{dimensions}

INSTRUCTIONS:

1. Create a Markdown comparison table with three columns:
   - Dimension
   - NVIDIA
   - AMD

2. Compare both companies across all four predefined
   dimensions.

3. Support every substantial factual claim with source
   citations using the exact source IDs provided,
   for example [A1] or [B2].

4. Use only information contained in the supplied context.

5. If information is unavailable for a company or dimension,
   explicitly write:
   "לא נמצא במקורות שנמסרו למודל".

6. Do not invent products, prices, customers, job openings,
   employee benefits or company promises.

7. Attribute marketing claims to the company that published them.

8. Only describe differences when the supplied sources
   provide a reasonable basis for comparison.

9. Do not interpret more detailed website information
   as proof that one company is better than the other.

10. After the table, write a concluding paragraph explaining:
    - What needs or customers each company may suit.
    - What additional information should be clarified
      before making a decision.
    - The limitations of comparing companies using
      their own published information.

11. Finish with a source list containing the source IDs
    and their corresponding URLs from the supplied context.

12. Do not introduce information from external knowledge.

SOURCE MATERIAL:
----------------------------------------
{context}
----------------------------------------

Produce a balanced, evidence-based comparison.
"""

    return prompt


In [104]:

# Comparison generation

from uuid import uuid4
from datetime import datetime, timezone
import os
import json
import time


def generate_comparison(context):

    print("Generating NVIDIA vs AMD comparison...")

    run_id = str(uuid4())
    start_time = time.time()

    # Build prompt without making an API call
    prompt = build_comparison_prompt(context)

    # Save the exact prompt used
    os.makedirs("data/prompts", exist_ok=True)

    prompt_path = (
        f"data/prompts/comparison_{run_id}.txt"
    )

    with open(prompt_path, "w", encoding="utf-8") as file:
        file.write(prompt)

    try:
        response = openai.with_options(
            max_retries=0
        ).chat.completions.create(
            model=MODEL,
            messages=[
                {
                    "role": "user",
                    "content": prompt
                }
            ]
        )

        if not response.choices:
            raise ValueError("Model returned no choices")

        comparison = response.choices[0].message.content

        if not comparison:
            raise ValueError("Model returned empty content")

        duration = round(time.time() - start_time, 2)

        # Save generated comparison
        os.makedirs("outputs/comparisons", exist_ok=True)

        output_path = (
            f"outputs/comparisons/"
            f"NVIDIA_AMD_{run_id}.md"
        )

        with open(output_path, "w", encoding="utf-8") as file:
            file.write(comparison)

        # Save run metadata
        metadata = {
            "run_id": run_id,
            "action": "company_comparison",
            "companies": ["NVIDIA", "AMD"],
            "model": MODEL,
            "timestamp": datetime.now(timezone.utc).isoformat(),
            "duration_seconds": duration,
            "status": "success",
            "prompt_path": prompt_path,
            "output_path": output_path
        }

        with open(
            f"outputs/comparisons/{run_id}_metadata.json",
            "w",
            encoding="utf-8"
        ) as file:
            json.dump(metadata, file, indent=4, ensure_ascii=False)

        print("\nComparison generated successfully!")
        print("Run ID:", run_id)
        print("Duration:", duration, "seconds")
        print("Saved to:", output_path)
        print("Prompt saved to:", prompt_path)

        return comparison, metadata

    except Exception as error:

        duration = round(time.time() - start_time, 2)

        print("\nComparison generation failed.")
        print("Error:", str(error))

        metadata = {
            "run_id": run_id,
            "action": "company_comparison",
            "model": MODEL,
            "duration_seconds": duration,
            "status": "failed",
            "error": str(error),
            "prompt_path": prompt_path
        }

        os.makedirs("outputs/comparisons", exist_ok=True)

        with open(
            f"outputs/comparisons/{run_id}_metadata.json",
            "w",
            encoding="utf-8"
        ) as file:
            json.dump(metadata, file, indent=4, ensure_ascii=False)

        raise


In [105]:

# Generate NVIDIA vs AMD comparison

comparison_result, comparison_metadata = generate_comparison(
    comparison_context
)

display(Markdown(comparison_result))


Generating NVIDIA vs AMD comparison...

Comparison generated successfully!
Run ID: 7d0dccf3-b4bc-4d25-9dc0-60d4011aee43
Duration: 8.04 seconds
Saved to: outputs/comparisons/NVIDIA_AMD_7d0dccf3-b4bc-4d25-9dc0-60d4011aee43.md
Prompt saved to: data/prompts/comparison_7d0dccf3-b4bc-4d25-9dc0-60d4011aee43.txt


להלן טבלת השוואה בין חברות NVIDIA ו-AMD בהתבסס על המקורות שסופקו:

### טבלת השוואה

| מימד | NVIDIA | AMD |
| :--- | :--- | :--- |
| **מוצרים ושירותים** | מייצרת שבבים, מערכות ותוכנה לבינה מלאכותית, כולל פלטפורמות מחשוב מואץ, Omniverse לדיגיטליזציה תעשייתית, וטכנולוגיות גרפיות (RTX, DLSS) [A3, A4]. | מציעה פורטפוליו הכולל מעבדי שרתים (EPYC), פתרונות מחשוב אדפטיביים, ורכיבי סיליקון להסקה (Inference) מבוססת בינה מלאכותית [B1, B3]. |
| **לקוחות יעד** | מפתחים, גיימרים, יוצרים, מעצבי ויזואליזציה מקצועיים ותעשיות כגון רכב, בריאות ורובוטיקה [A2, A4]. | מרכזי נתונים, עננים היפר-סקייל (Hyperscale), מחשבי-על, עסקים קטנים וספקי שירותים מארחים [B3, B4]. |
| **יכולות בינה מלאכותית ומחשוב** | מתמקדת ב"מפעלי AI" עם תשתית הכוללת חומרה, תוכנה (NIM, NeMo) ומודלי עולם (Cosmos) לביצועים מואצים [A3, A4]. | מתמקדת בגישה פתוחה וגמישה, ומציעה את הפורטפוליו היחיד הכולל שילוב של CPU, GPU ומחשוב אדפטיבי [B1]. |
| **קריירה וסביבת עבודה** | מתוארת כסביבה המעודדת שיתוף פעולה, שונות ואינדיבידואליות, בה "הפרויקט הוא הבוס" [A2]. | מדגישה עבודה משמעותית, חדשנות וטיפוח כוח עבודה כוללני, עם תוכניות סטודנטים ייעודיות [B2, B4]. |

---

### סיכום
*   **התאמה לצרכים ולקוחות:** חברת **NVIDIA** נראית כמתאימה במיוחד לארגונים המחפשים פתרונות מקצה לקצה בתחומי הבינה המלאכותית היוצרת, פיתוח גרפי מורכב, וסימולציות דיגיטליות (Digital Twins). לעומת זאת, **AMD** מציעה יתרון ללקוחות הזקוקים לגמישות רבה יותר, תאימות למערכות אקולוגיות פתוחות ופתרונות מגוונים המשלבים CPU, GPU ומחשוב אדפטיבי עבור מרכזי נתונים ושירותי ענן.
*   **מידע נדרש להשלמה:** כדי לקבל החלטה מושכלת, מומלץ לבחון נתונים טכניים השוואתיים (Benchmark) עבור יישומים ספציפיים, לבדוק את התאימות של התוכנות הקיימות בארגון לתשתיות של כל אחת מהחברות, ולברר עלויות כוללות לבעלות (TCO) המותאמות לדרישות העסקיות הספציפיות.
*   **מגבלות ההשוואה:** השוואה זו מבוססת אך ורק על חומרים שיווקיים ומידע שפורסם על ידי החברות עצמן. מידע כזה נוטה להדגיש את היתרונות והחוזקות של כל חברה, ואינו מספק אובייקטיביות מלאה או השוואה בלתי-תלויה של ביצועים בשטח, מה שעלול להשאיר פערים משמעותיים בהבנת היכולות בפועל תחת עומסי עבודה אמיתיים.

---

### רשימת מקורות
*   **A1:** https://www.nvidia.com/
*   **A2:** https://www.nvidia.com/en-eu/about-nvidia/careers/
*   **A3:** https://www.nvidia.com/en-eu/about-nvidia/
*   **A4:** https://www.nvidia.com/en-eu/software/
*   **B1:** https://www.amd.com/en.html
*   **B2:** https://www.amd.com/en/corporate/careers.html
*   **B3:** https://www.amd.com/en/products/processors/server/epyc.html
*   **B4:** https://www.amd.com/en/corporate.html

In [108]:

# comparison validation

def validate_comparison(comparison, allowed_source_ids, sources):

    parts = re.split(
        r"(?im)^#{1,6}\s*רשימת מקורות\s*$",
        comparison,
        maxsplit=1
    )

    main_content = parts[0]
    source_section = parts[1] if len(parts) > 1 else ""

    # Extract citations from comparison body
    cited_ids = set(
        re.findall(r"\[(A\d+|B\d+)\]", main_content)
    )

    invalid_ids = cited_ids - set(allowed_source_ids)

    # Extract source IDs and URLs (plain URLs or Markdown links)
    source_entries = re.findall(
        r"(?m)^\s*[-*]\s*\*\*(A\d+|B\d+):\*\*\s*"
        r"(?:\[)?(https?://\S+?)(?:\])?\s*$",
        source_section
    )

    listed_ids = {source_id for source_id, url in source_entries}

    expected_urls = {
        source["source_id"]: source["url"]
        for source in sources
    }

    incorrect_urls = [
        {
            "source_id": source_id,
            "provided": url,
            "expected": expected_urls.get(source_id)
        }
        for source_id, url in source_entries
        if expected_urls.get(source_id) != url
    ]

    # Only cited sources must appear in the source list
    missing_sources = cited_ids - listed_ids

    required_dimensions = [
        "מוצרים ושירותים",
        "לקוחות יעד",
        "יכולות בינה מלאכותית ומחשוב",
        "קריירה וסביבת עבודה"
    ]

    missing_dimensions = [
        dimension
        for dimension in required_dimensions
        if dimension not in main_content
    ]

    results = {
        "citations_valid": len(invalid_ids) == 0,
        "invalid_ids": sorted(invalid_ids),
        "source_list_complete": len(missing_sources) == 0,
        "missing_sources": sorted(missing_sources),
        "source_urls_valid": len(incorrect_urls) == 0,
        "incorrect_urls": incorrect_urls,
        "dimensions_complete": len(missing_dimensions) == 0,
        "missing_dimensions": missing_dimensions
    }

    results["overall_passed"] = all([
        results["citations_valid"],
        results["source_list_complete"],
        results["source_urls_valid"],
        results["dimensions_complete"]
    ])

    print("========== COMPARISON VALIDATION ==========")

    for name, key in [
        ("Citation validation", "citations_valid"),
        ("Source list completeness", "source_list_complete"),
        ("Source URL validation", "source_urls_valid"),
        ("Four dimensions present", "dimensions_complete")
    ]:
        print(f"{name}: {'PASSED' if results[key] else 'FAILED'}")

    print("\nCited IDs:", sorted(cited_ids))
    print("Listed IDs:", sorted(listed_ids))
    print("Missing cited sources:", sorted(missing_sources))
    print("Invalid IDs:", sorted(invalid_ids))

    print(
        "\nOverall result:",
        "PASSED" if results["overall_passed"] else "FAILED"
    )

    return results


In [109]:

comparison_validation = validate_comparison(
    comparison_result,
    comparison_manifest["included_sources"],
    combined_sources
)


========== COMPARISON VALIDATION ==========
Citation validation: PASSED
Source list completeness: PASSED
Source URL validation: PASSED
Four dimensions present: PASSED

Cited IDs: ['A2', 'B1']
Listed IDs: ['A1', 'A2', 'A3', 'A4', 'B1', 'B2', 'B3', 'B4']
Missing cited sources: []
Invalid IDs: []

Overall result: PASSED


## Streaming implimitation

In [110]:

from IPython.display import display, Markdown


def generate_brochure_stream(company_name, context, audience):
    """
    Generate and display a brochure incrementally.
    Return the accumulated text and its completion status.
    """

    print(f"Generating {company_name} brochure for {audience}...")

    prompt = build_brochure_prompt(
        company_name,
        audience,
        context
    )

    accumulated_text = ""
    completed = False

    display_handle = display(
        Markdown(""),
        display_id=True
    )

    try:
        stream = openai.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "user", "content": prompt}
            ],
            stream=True
        )

        for chunk in stream:

            if not chunk.choices:
                continue

            choice = chunk.choices[0]
            delta = choice.delta

            content = getattr(delta, "content", None)

            if isinstance(content, str) and content:
                accumulated_text += content
                display_handle.update(
                    Markdown(accumulated_text)
                )

            if choice.finish_reason == "stop":
                completed = True

    except Exception as error:
        print(f"\nStreaming interrupted: {error}")

    status = "completed" if completed else "partial"

    print(f"\nStatus: {status}")
    print(f"Characters collected: {len(accumulated_text)}")

    return accumulated_text, status


In [111]:

# Streaming event processor

def process_stream_events(stream, on_update=None):
    """
    Process streaming events, accumulate text and handle interruptions.
    Can be tested using simulated events without API calls.
    """

    accumulated_text = ""
    completed = False
    error_message = None

    try:
        for chunk in stream:

            # Handle None events
            if chunk is None:
                continue

            # Handle missing or empty choices
            choices = getattr(chunk, "choices", None)

            if not choices:
                continue

            choice = choices[0]

            # Extract text safely
            delta = getattr(choice, "delta", None)

            if delta is not None:
                content = getattr(delta, "content", None)

                if isinstance(content, str) and content:
                    accumulated_text += content

                    if on_update:
                        on_update(accumulated_text)

            # Detect successful completion
            if getattr(choice, "finish_reason", None) == "stop":
                completed = True

    except Exception as error:
        error_message = str(error)

    return {
        "text": accumulated_text,
        "status": "completed" if completed else "partial",
        "error": error_message
    }


In [112]:

# Create simulated streaming events

from types import SimpleNamespace

def create_fake_chunk(content=None, finish_reason=None, choices=True):

    if choices is False:
        return SimpleNamespace(choices=[])

    if choices is None:
        return SimpleNamespace(choices=None)

    delta = SimpleNamespace(content=content)

    choice = SimpleNamespace(
        delta=delta,
        finish_reason=finish_reason
    )

    return SimpleNamespace(choices=[choice])


simulated_events = [
    create_fake_chunk("Hello "),
    create_fake_chunk(None),
    None,
    create_fake_chunk("this is "),
    create_fake_chunk("a streaming test."),
    create_fake_chunk(None, choices=False),
    create_fake_chunk(None, finish_reason="stop")
]

print("Simulated events created:", len(simulated_events))


Simulated events created: 7


In [114]:

# Test simulated streaming

simulated_result = process_stream_events(
    simulated_events
)

print("Accumulated text:")
print(simulated_result["text"])

print("\nStatus:", simulated_result["status"])

# Verify exact output
expected_text = "Hello this is a streaming test."

assert simulated_result["text"] == expected_text
assert simulated_result["status"] == "completed"
assert "None" not in simulated_result["text"]

print("\nAll simulated streaming checks passed!")


Accumulated text:
Hello this is a streaming test.

Status: completed

All simulated streaming checks passed!


In [115]:

# Test interrupted streaming

def interrupted_stream():

    yield create_fake_chunk("This is ")

    yield create_fake_chunk("an interrupted ")

    raise ConnectionError("Simulated connection failure")


interrupted_result = process_stream_events(
    interrupted_stream()
)

print("Accumulated text:", interrupted_result["text"])
print("Status:", interrupted_result["status"])
print("Error:", interrupted_result["error"])

# Verify interruption handling

assert interrupted_result["text"] == "This is an interrupted "
assert interrupted_result["status"] == "partial"
assert interrupted_result["error"] is not None

print("\nAll interruption checks passed!")


Accumulated text: This is an interrupted 
Status: partial
Error: Simulated connection failure

All interruption checks passed!


In [157]:
def generate_brochure_stream(company_name, context, audience):

    print(f"Generating {company_name} brochure for {audience}...")

    run_id = str(uuid4())
    start_time = time.time()

    prompt = build_brochure_prompt(
        company_name,
        audience,
        context
    )

    display_handle = display(
        Markdown(""),
        display_id=True
    )

    # Initialize before the API call
    brochure = ""
    status = "partial"
    result = {
        "text": "",
        "status": "partial",
        "error": None
    }

    # Keep a copy of every successfully received text update
    def update_display(text):
        nonlocal brochure

        brochure = text
        display_handle.update(Markdown(text))

    try:
        stream = openai.with_options(
            max_retries=0
        ).chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "user", "content": prompt}
            ],
            stream=True
        )

        # Process events and display text gradually
        result = process_stream_events(
            stream,
            on_update=update_display
        )

        brochure = result["text"]
        status = result["status"]

    except Exception as error:
        # Preserve any text already accumulated
        status = "partial"

        result = {
            "text": brochure,
            "status": status,
            "error": str(error)
        }

    duration = round(time.time() - start_time, 2)

    # Save the streamed result
    os.makedirs("outputs/streaming", exist_ok=True)

    output_path = (
        f"outputs/streaming/"
        f"{company_name}_{audience}_{run_id}.md"
    )

    with open(output_path, "w", encoding="utf-8") as file:
        file.write(brochure)

    # Save execution metadata
    metadata = {
        "run_id": run_id,
        "company": company_name,
        "audience": audience,
        "model": MODEL,
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "duration_seconds": duration,
        "status": status,
        "output_path": output_path,
        "characters": len(brochure),
        "error": result.get("error")
    }

    metadata_path = (
        f"outputs/streaming/{run_id}_metadata.json"
    )

    with open(metadata_path, "w", encoding="utf-8") as file:
        json.dump(
            metadata,
            file,
            indent=4,
            ensure_ascii=False
        )

    print("\nStreaming finished.")
    print("Status:", status)
    print("Run ID:", run_id)
    print("Duration:", duration, "seconds")
    print("Saved to:", output_path)
    print("Metadata saved to:", metadata_path)

    return brochure, status

In [117]:

# Real streaming demonstration

streamed_brochure, streaming_status = generate_brochure_stream(
    company_name="NVIDIA",
    context=nvidia_context,
    audience="customers"
)

print("\nFinal streaming status:", streaming_status)
print("Final word count:", len(
    re.split(
        r"(?im)^#{1,6}\s*מקורות\s*$",
        streamed_brochure,
        maxsplit=1
    )[0].split()
))


Generating NVIDIA brochure for customers...


# פתרונות מחשוב מואץ מבית NVIDIA: העתיד של הבינה המלאכותית

NVIDIA מובילה את מהפכת המחשוב המואץ במטרה לפתור את האתגרים המורכבים ביותר בעולם הטכנולוגי [A3]. החברה מפתחת פלטפורמות, שבבים ותוכנות מתקדמות המאפשרות לתעשיות שלמות לעבור טרנספורמציה דיגיטלית ולפעול בצורה מהירה, חכמה ויעילה יותר [A2, A3].

### תחומי פעילות ופתרונות
NVIDIA מציעה מגוון רחב של כלים ושירותים המותאמים למגוון צרכים עסקיים ומדעיים:
* **בינה מלאכותית (AI):** פיתוח תשתיות ל"מפעלי AI", המאפשרות אימון מודלים של שפה גדולה (LLM), ניתוח נתונים מאסיבי לשיפור החלטות עסקיות ושימוש ב-NIM Microservices [A2, A3, A4].
* **מרכזי נתונים (Data Center):** פלטפורמות כדוגמת Vera, המשלבות מעבדים ופתרונות רישות לביצועים גבוהים במשימות סימולציה, מודלים מדעיים והסקה (Inference) [A1].
* **דיגיטציה תעשייתית:** פלטפורמת Omniverse ליצירת תאומים דיגיטליים (Digital Twins) המאפשרים חיבור בין העולם הפיזי לדיגיטלי [A3, A4].
* **רכב אוטונומי:** פתרונות המעניקים לכלי רכב את היכולת לראות, ללמוד ולנווט בתרחישים מורכבים [A2].
* **בריאות ומדע:** כלים להאצת מחקר בתחומי מדעי החיים, גילוי תרופות וסימולציות רפואיות [A1, A4].

### מקרי בוחן ושימושים מעשיים
הפתרונות של החברה מיושמים במגוון סקטורים:
1. **שיפור גרפיקה:** שימוש ב-RTX ו-DLSS 3 ליצירת ריאליזם חסר תקדים [A3].
2. **קיימות:** התייעלות אנרגטית במרכזי נתונים באמצעות מחשוב מואץ [A3].
3. **מחקר מדעי:** האצת גילויים בתחומי הכימיה והפיזיקה באמצעות Nemotron ופלטפורמות מחשוב קוונטי [A1].

### קהל יעד
הפתרונות של NVIDIA מיועדים לארגונים וחברות מכל סדר גודל המבקשים לאמץ טכנולוגיות פורצות דרך, לרבות חברות בתחומי ה-IT, הרכב, הרפואה והנדסה [A2, A3].

### מה על הלקוח להבהיר לפני פנייה?
בטרם פנייה לחברה, על לקוחות פוטנציאליים לבחון:
* מהם האתגרים הטכנולוגיים הספציפיים שהארגון מבקש לפתור (למשל: צורך בניתוח נתונים, סימולציות או מודלים של בינה מלאכותית)?
* מהן הדרישות התשתיתיות והחישוביות של הארגון [A2, A4]?

*מידע בנוגע למחירים, תנאי התקשרות או הצעות מחיר פרטניות – לא נמצא במקורות שנמסרו למודל.*

### מקורות
[A1] https://www.nvidia.com/
[A2] https://www.nvidia.com/en-eu/about-nvidia/careers/
[A3] https://www.nvidia.com/en-eu/about-nvidia/
[A4] https://www.nvidia.com/en-eu/software/


Streaming finished.
Status: completed
Run ID: 6d82521d-c8ff-47da-bb5a-31afd2fa3bd7
Duration: 9.29 seconds
Saved to: outputs/streaming/NVIDIA_customers_6d82521d-c8ff-47da-bb5a-31afd2fa3bd7.md
Metadata saved to: outputs/streaming/6d82521d-c8ff-47da-bb5a-31afd2fa3bd7_metadata.json

Final streaming status: completed
Final word count: 286


In [118]:

# Validate the streamed NVIDIA customer brochure

stream_validation = validate_brochure(
    streamed_brochure,
    nvidia_manifest["included_sources"]
)

assert streaming_status == "completed"
assert stream_validation["word_count_valid"]
assert len(stream_validation["invalid_ids"]) == 0

print("\nStreaming brochure validation: PASSED")


Word count: 286
Word count valid: True

Allowed source IDs: ['A1', 'A2', 'A3', 'A4']
Cited source IDs: ['A1', 'A2', 'A3']
Invalid source IDs: []

Citation ID validation: True

Streaming brochure validation: PASSED


In [119]:

import os

print("Current directory:")
print(os.getcwd())

print("\nFolder checks:")

for folder in [
    "../data",
    "../logs",
    "data",
    "logs",
    "outputs"
]:
    print(f"{folder}: {os.path.exists(folder)}")


Current directory:
c:\Users\Diana\Desktop\ai-projects\assignment1-brochures\notebooks

Folder checks:
../data: True
../logs: True
data: False
logs: False
outputs: False


Baseline prompt experiment

In [ ]:
# This is our baseline: a simpler prompt without the detailed audience-specific instructions 
# and additional controls of your improved build_brochure_prompt.
def build_baseline_prompt(company_name, audience, context):

    prompt = f"""
Write a company brochure in Hebrew about {company_name}.

Target audience: {audience}

Use the supplied company information to write
a clear and informative brochure.

Requirements:
- Write 200–300 words, excluding the source list.
- Include an introduction and relevant sections.
- Explain the company's activities and products.
- Adapt the content to the target audience.
- Use factual information from the supplied sources.
- Cite factual claims using source IDs such as [A1].
- Do not invent information.
- Mention when information is unavailable.
- Include a sources section with source IDs and URLs.

Company information:
--------------------
{context}
--------------------

Write the complete brochure in Hebrew.
"""
    return prompt

In [127]:
# Generate and save the baseline brochure 
from uuid import uuid4
from pathlib import Path
from datetime import datetime
import time

# Generate baseline brochure
run_id = str(uuid4())

print("Generating NVIDIA Customers baseline...")

start_time = time.time()

response = openai.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": baseline_prompt}
    ]
)

baseline_brochure = response.choices[0].message.content

duration = round(time.time() - start_time, 2)

# Save output
output_dir = Path("outputs/baseline")
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "NVIDIA_customers_baseline.md"

output_path.write_text(
    baseline_brochure,
    encoding="utf-8"
)

# Save exact prompt
prompt_dir = Path("data/prompts")
prompt_dir.mkdir(parents=True, exist_ok=True)

prompt_path = prompt_dir / "NVIDIA_customers_baseline.txt"

prompt_path.write_text(
    baseline_prompt,
    encoding="utf-8"
)

print("\nBaseline generated successfully!")
print("Run ID:", run_id)
print("Duration:", duration, "seconds")
print("Saved to:", output_path)
print("Prompt saved to:", prompt_path)
print("\nBaseline brochure:\n")
print(baseline_brochure)


Generating NVIDIA Customers baseline...

Baseline generated successfully!
Run ID: d6149c02-7f96-4d30-b5f6-f14663fe2e71
Duration: 7.58 seconds
Saved to: outputs\baseline\NVIDIA_customers_baseline.md
Prompt saved to: data\prompts\NVIDIA_customers_baseline.txt

Baseline brochure:

### NVIDIA: המנוע של עידן הבינה המלאכותית

**אודות NVIDIA**
NVIDIA מובילה את מהפכת המחשוב המואץ במטרה לפתור את האתגרים המורכבים ביותר בעולם [A3]. כחלוצה בתחום הבינה המלאכותית (AI) והתאומים הדיגיטליים, החברה מספקת טכנולוגיות המשנות תעשיות שלמות – החל משירותי בריאות ועד לנהיגה אוטונומית ומחשוב קוונטי [A2, A3]. אנו מגדירים מחדש את גבולות האפשר ומהווים את "המנוע" מאחורי מפעלי ה-AI של העתיד [A3].

**הפתרונות שלנו**
*   **מרכזי נתונים ומחשוב עתיר ביצועים (HPC):** פלטפורמות ה-Vera של NVIDIA מאיצות גילויים מדעיים, סימולציות ומודלים מורכבים [A1]. אנו משלבים חומרה מתקדמת, מעבדים מרכזיים (CPUs) וטכנולוגיות רישות פורצות דרך כדי לספק יעילות אנרגטית ותפוקה גבוהה [A1, A3].
*   **בינה מלאכותית יוצרת (Generative AI):** בעזרת כלי

In [128]:
# Validate baseline brochure

baseline_validation = validate_brochure(
    baseline_brochure,
    nvidia_manifest["included_sources"]
)

print("\n========== BASELINE VALIDATION ==========")

print("Word count valid:",
      baseline_validation["word_count_valid"])

print("Invalid citations:",
      baseline_validation["invalid_ids"])

print("All citations valid:",
      len(baseline_validation["invalid_ids"]) == 0)

Word count: 289
Word count valid: True

Allowed source IDs: ['A1', 'A2', 'A3', 'A4']
Cited source IDs: ['A1', 'A2', 'A3', 'A4']
Invalid source IDs: []

Citation ID validation: True

========== BASELINE VALIDATION ==========
Word count valid: True
Invalid citations: set()
All citations valid: True


## Testing

In [ ]:
# Test one valid selection
test_base_url = "https://www.nvidia.com/"

test_candidates = [
    "https://www.nvidia.com/en-eu/about-nvidia/",
    "https://www.nvidia.com/en-eu/software/",
    "https://www.nvidia.com/en-eu/about-nvidia/careers/",
    "https://www.nvidia.com/en-eu/privacy/",
]

valid_response = json.dumps({
    "links": [
        {
            "type": "about",
            "url": "https://www.nvidia.com/en-eu/about-nvidia/",
            "reason": "Company information"
        },
        {
            "type": "software",
            "url": "https://www.nvidia.com/en-eu/software/",
            "reason": "Product information"
        }
    ]
})

result = validate_link_selection(
    valid_response,
    test_base_url,
    test_candidates
)

print("VALID SELECTION ACCEPTED")
print(json.dumps(result, indent=2))

VALID SELECTION ACCEPTED
{
  "links": [
    {
      "type": "about",
      "url": "https://www.nvidia.com/en-eu/about-nvidia/",
      "reason": "Company information"
    },
    {
      "type": "software",
      "url": "https://www.nvidia.com/en-eu/software/",
      "reason": "Product information"
    }
  ]
}


In [137]:
# Test invalid link selections
import json

def make_link(url):
    return {
        "type": "information",
        "url": url,
        "reason": "Relevant company information"
    }

valid_url = "https://www.nvidia.com/en-eu/about-nvidia/"
second_url = "https://www.nvidia.com/en-eu/software/"
privacy_url = "https://www.nvidia.com/en-eu/privacy/"

failure_cases = {
    "Malformed JSON": '{"links": [',

    "URL outside candidate list": json.dumps({
        "links": [
            make_link("https://www.nvidia.com/not-in-candidates/")
        ]
    }),

    "Duplicate URLs": json.dumps({
        "links": [
            make_link(valid_url),
            make_link(valid_url)
        ]
    }),

    "Forbidden URL mixed with valid URLs": json.dumps({
        "links": [
            make_link(valid_url),
            make_link(privacy_url),
            make_link(second_url)
        ]
    })
}

test_results = []

for test_name, raw_response in failure_cases.items():

    try:
        validated = validate_link_selection(
            raw_response,
            test_base_url,
            test_candidates
        )

        result = "FAIL"
        reason = "Invalid selection was accepted"

    except ValueError as error:
        result = "PASS"
        reason = str(error)

    test_results.append({
        "test": test_name,
        "result": result,
        "reason": reason,
        "fetch_calls": 0
    })

    print(f"\n{test_name}: {result}")
    print("Reason:", reason)

assert all(
    result["result"] == "PASS"
    for result in test_results
), "One or more failure tests did not pass"

print("\nAll four invalid-selection tests passed!")


Malformed JSON: PASS
Reason: Malformed JSON: Expecting value: line 1 column 12 (char 11)

URL outside candidate list: PASS
Reason: URL not in candidate list: https://www.nvidia.com/not-in-candidates/

Duplicate URLs: PASS
Reason: Duplicate URL: https://www.nvidia.com/en-eu/about-nvidia/

Forbidden URL mixed with valid URLs: PASS
Reason: Forbidden page selected: https://www.nvidia.com/en-eu/privacy/

All four invalid-selection tests passed!


In [ ]:
# updated so that it delegates validation to our new, reusable validate_link_selection() function
# if Gemini returns an invalid selection of webpage links, the program rejects the entire response
#  before fetching any webpages
def select_relevant_links(url):
    """
    Ask Gemini to select links, validate the complete response,
    and save rejection details if validation fails.
    """

    # Build and save the exact candidate list used for this call
    user_prompt = get_links_user_prompt(url)
    candidate_links = list(normalized_candidate_links)

    # Gemini API call
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        response_format={"type": "json_object"}
    )

    raw_text = response.choices[0].message.content

    try:
        selected = validate_link_selection(
            raw_text,
            url,
            candidate_links
        )

        print("Gemini selection validated successfully.")
        return selected

    except ValueError as error:
        # Save the rejected response and explanation
        os.makedirs("logs", exist_ok=True)

        rejection = {
            "company_url": url,
            "raw_response": raw_text,
            "candidate_links": candidate_links,
            "rejection_reason": str(error)
        }

        with open(
            "logs/link_selection_rejection.json",
            "w",
            encoding="utf-8"
        ) as file:
            json.dump(
                rejection,
                file,
                indent=4,
                ensure_ascii=False
            )

        raise ValueError(
            f"Gemini link selection rejected: {error}"
        ) from error


Test validation before fetching

In [ ]:

from unittest.mock import Mock

workflow_test_results = []

for test_name, raw_response in failure_cases.items():
    mock_fetch = Mock()
    rejected = False
    reason = None

    try:
        selected = validate_link_selection(
            raw_response,
            test_base_url,
            test_candidates
        )

        # This runs only if validation succeeds.
        mock_fetch("A", test_base_url, selected)

    except ValueError as error:
        rejected = True
        reason = str(error)

    fetch_calls = mock_fetch.call_count
    passed = rejected and fetch_calls == 0

    workflow_test_results.append({
        "test": test_name,
        "status": "PASS" if passed else "FAIL",
        "reason": reason,
        "fetch_calls": fetch_calls
    })

    print(f"{test_name}: {'PASS' if passed else 'FAIL'}")
    print(f"  Fetch calls: {fetch_calls}")
    print(f"  Reason: {reason}\n")

assert all(
    result["status"] == "PASS"
    for result in workflow_test_results
)

print("All workflow safety tests passed!")

Malformed JSON: PASS
  Fetch calls: 0
  Reason: Malformed JSON: Expecting value: line 1 column 12 (char 11)

URL outside candidate list: PASS
  Fetch calls: 0
  Reason: URL not in candidate list: https://www.nvidia.com/not-in-candidates/

Duplicate URLs: PASS
  Fetch calls: 0
  Reason: Duplicate URL: https://www.nvidia.com/en-eu/about-nvidia/

Forbidden URL mixed with valid URLs: PASS
  Fetch calls: 0
  Reason: Forbidden page selected: https://www.nvidia.com/en-eu/privacy/

All workflow safety tests passed!


In [144]:
# Save the test results
import json
from pathlib import Path

tests_dir = Path("tests")
tests_dir.mkdir(exist_ok=True)

results_path = tests_dir / "link_failure_results.json"

with open(results_path, "w", encoding="utf-8") as file:
    json.dump(
        {
            "test_type": "validation_before_fetch_simulation",
            "total_tests": len(workflow_test_results),
            "passed": sum(
                result["status"] == "PASS"
                for result in workflow_test_results
            ),
            "results": workflow_test_results
        },
        file,
        indent=4,
        ensure_ascii=False
    )

print("Results saved to:", results_path.resolve())

Results saved to: C:\Users\Diana\Desktop\ai-projects\assignment1-brochures\tests\link_failure_results.json


Test empty and failed webpage sources

In [ ]:
# Simulate webpage responses
from unittest.mock import patch

test_homepage = "https://example.com/"

test_selection = {
    "links": [
        {
            "type": "about",
            "url": "https://example.com/about",
            "reason": "Simulate empty content"
        },
        {
            "type": "careers",
            "url": "https://example.com/careers",
            "reason": "Simulate connection failure"
        }
    ]
}

def fake_fetch_website_contents(url):
    if url == test_homepage:
        return "Example Company\n\nThis is valid homepage content."

    if url.endswith("/about"):
        return ""

    if url.endswith("/careers"):
        raise ConnectionError("Simulated connection failure")

    raise AssertionError(f"Unexpected URL: {url}")


# Temporarily replace the real website-fetching function
with patch.dict(
    globals(),
    {"fetch_website_contents": fake_fetch_website_contents}
):
    fetch_company_sources(
        "TEST",
        test_homepage,
        test_selection
    )

Fetching TEST1: https://example.com/
Fetching TEST2: https://example.com/about
Fetching TEST3: https://example.com/careers
Failed to fetch TEST3: Simulated connection failure

Saved 3 sources to data/sources/TEST_sources.json


In [ ]:
# Verify and save results
import json
from pathlib import Path

# Load the test results already saved by fetch_company_sources()
test_file = Path("data/sources/TEST_sources.json")

with open(test_file, "r", encoding="utf-8") as file:
    test_sources = json.load(file)

expected_statuses = ["ok", "empty", "failed"]
actual_statuses = [source["status"] for source in test_sources]

for source in test_sources:
    print(
        f"{source['source_id']}: "
        f"{source['status']} | "
        f"text length: {len(source['text'])}"
    )

assert actual_statuses == expected_statuses, (
    f"Expected {expected_statuses}, got {actual_statuses}"
)

print("\nAll source status tests passed!")

# Save verified results in the tests folder
destination = Path("tests/source_status_results.json")
destination.parent.mkdir(parents=True, exist_ok=True)

with open(destination, "w", encoding="utf-8") as file:
    json.dump({
        "test_type": "source_status_handling",
        "expected_statuses": expected_statuses,
        "actual_statuses": actual_statuses,
        "passed": True,
        "sources": test_sources
    }, file, indent=4, ensure_ascii=False)

print("Results saved to:", destination)

TEST1: ok | text length: 31
TEST2: empty | text length: 0
TEST3: failed | text length: 0

All source status tests passed!
Results saved to: tests\source_status_results.json


Testing oversized webpage content and the context character budget

In [150]:
# Simulate one oversized source and two unusable sources
long_test_sources = [
    {
        "source_id": "T1",
        "company_id": "TEST",
        "url": "https://example.com/long",
        "title": "Long Test Page",
        "text": "This is a long webpage. " * 500,
        "status": "ok"
    },
    {
        "source_id": "T2",
        "company_id": "TEST",
        "url": "https://example.com/empty",
        "title": "Empty Test Page",
        "text": "",
        "status": "empty"
    },
    {
        "source_id": "T3",
        "company_id": "TEST",
        "url": "https://example.com/failed",
        "title": "Failed Test Page",
        "text": "",
        "status": "failed"
    }
]

# Use a small budget to force truncation
test_budget = 500

test_context, test_manifest = build_context(
    long_test_sources,
    text_budget=test_budget,
    output_name="TEST_oversized"
)

# Verify that the context respects the budget
assert len(test_context) <= test_budget

# Only the valid source should be included
assert test_manifest["included_sources"] == ["T1"]

# The long source must be truncated
assert len(test_manifest["truncated_sources"]) == 1
assert test_manifest["truncated_sources"][0]["source_id"] == "T1"

# Invalid sources must be omitted
omitted_ids = {
    source["source_id"]
    for source in test_manifest["omitted_sources"]
}
assert omitted_ids == {"T2", "T3"}

# Check that source metadata is preserved
assert "## Source T1" in test_context
assert "https://example.com/long" in test_context

print("\nAll oversized-source tests passed!")
print("Original text length:", len(long_test_sources[0]["text"]))
print("Final context length:", len(test_context))
print("Budget:", test_budget)
print("Included:", test_manifest["included_sources"])
print("Omitted:", sorted(omitted_ids))

Context created: TEST_oversized
Characters used: 500/500
Included sources: ['T1']
Truncated sources: 1
Omitted sources: 2

All oversized-source tests passed!
Original text length: 12000
Final context length: 500
Budget: 500
Included: ['T1']
Omitted: ['T2', 'T3']


In [151]:
# save results
from pathlib import Path
import json

test_results = {
    "test_type": "context_budget_and_source_filtering",
    "status": "PASS",
    "original_text_length": len(long_test_sources[0]["text"]),
    "context_length": len(test_context),
    "character_budget": test_budget,
    "included_sources": test_manifest["included_sources"],
    "truncated_sources": test_manifest["truncated_sources"],
    "omitted_sources": test_manifest["omitted_sources"],
    "checks": {
        "within_budget": len(test_context) <= test_budget,
        "valid_source_included": test_manifest["included_sources"] == ["T1"],
        "oversized_source_truncated": len(test_manifest["truncated_sources"]) == 1,
        "invalid_sources_excluded": {
            item["source_id"]
            for item in test_manifest["omitted_sources"]
        } == {"T2", "T3"},
        "source_metadata_preserved": (
            "## Source T1" in test_context
            and "https://example.com/long" in test_context
        )
    }
}

assert all(test_results["checks"].values())

output_path = Path("tests/context_budget_results.json")
output_path.parent.mkdir(parents=True, exist_ok=True)

with open(output_path, "w", encoding="utf-8") as file:
    json.dump(test_results, file, indent=4, ensure_ascii=False)

print("All checks passed!")
print("Saved to:", output_path)

All checks passed!
Saved to: tests\context_budget_results.json


In [154]:
# Test simulated streaming events
from pathlib import Path
import json

# Simulate different types of streaming events
fake_events = [
    create_fake_chunk("Hello "),
    None,
    create_fake_chunk(None),
    create_fake_chunk(""),
    create_fake_chunk(choices=False),
    create_fake_chunk(choices=None),
    create_fake_chunk("world!"),
    create_fake_chunk(finish_reason="stop")
]

# Record each progressive display update
updates = []

result = process_stream_events(
    fake_events,
    on_update=lambda text: updates.append(text)
)

expected_text = "Hello world!"

# Verify the accumulated text and status
assert result["text"] == expected_text
assert result["status"] == "completed"
assert result["error"] is None
assert updates == ["Hello ", "Hello world!"]
assert "None" not in result["text"]

# Save the exact accumulated text
output_dir = Path("tests")
output_dir.mkdir(parents=True, exist_ok=True)

text_path = output_dir / "streaming_simulated_output.txt"
text_path.write_text(result["text"], encoding="utf-8")

# Verify the saved file matches exactly
assert text_path.read_text(encoding="utf-8") == expected_text

print("Status:", result["status"])
print("Accumulated text:", repr(result["text"]))
print("Progressive updates:", updates)
print("Saved text:", repr(text_path.read_text(encoding="utf-8")))
print("\nAll simulated streaming tests passed!")

Status: completed
Accumulated text: 'Hello world!'
Progressive updates: ['Hello ', 'Hello world!']
Saved text: 'Hello world!'

All simulated streaming tests passed!


In [ ]:
# Test an interrupted stream
from pathlib import Path
import json

# Run the simulated interrupted stream
interrupted_updates = []

interrupted_result = process_stream_events(
    interrupted_stream(),
    on_update=lambda text: interrupted_updates.append(text)
)

expected_partial_text = "This is an interrupted "

# Verify the interruption was handled correctly
assert interrupted_result["text"] == expected_partial_text
assert interrupted_result["status"] == "partial"
assert interrupted_result["error"] == "Simulated connection failure"
assert interrupted_updates == [
    "This is ",
    "This is an interrupted "
]
assert "None" not in interrupted_result["text"]

# Save the partial text exactly as accumulated
output_dir = Path("tests")
output_dir.mkdir(parents=True, exist_ok=True)

text_path = output_dir / "streaming_interrupted_output.txt"
text_path.write_text(
    interrupted_result["text"],
    encoding="utf-8"
)

# Save the interruption metadata
metadata_path = output_dir / "streaming_interrupted_results.json"
metadata_path.write_text(
    json.dumps(interrupted_result, indent=4, ensure_ascii=False),
    encoding="utf-8"
)

# Verify that the saved text matches the accumulated text
assert text_path.read_text(encoding="utf-8") == expected_partial_text

print("Status:", interrupted_result["status"])
print("Accumulated text:", repr(interrupted_result["text"]))
print("Error:", interrupted_result["error"])
print("Progressive updates:", interrupted_updates)
print("Saved text:", repr(text_path.read_text(encoding="utf-8")))

print("\nAll interrupted-stream tests passed!")

Status: partial
Accumulated text: 'This is an interrupted '
Error: Simulated connection failure
Progressive updates: ['This is ', 'This is an interrupted ']
Saved text: 'This is an interrupted '

All interrupted-stream tests passed!


In [160]:
# Test the complete streaming workflow
from unittest.mock import Mock
from types import SimpleNamespace
from pathlib import Path
import json

# Track calls to the fake API
fake_create = Mock(side_effect=lambda **kwargs: interrupted_stream())

fake_client = SimpleNamespace(
    with_options=lambda **kwargs: SimpleNamespace(
        chat=SimpleNamespace(
            completions=SimpleNamespace(create=fake_create)
        )
    )
)

# Patch the exact namespace used by the function
function_globals = generate_brochure_stream.__globals__
original_client = function_globals["openai"]

try:
    function_globals["openai"] = fake_client

    # Confirm replacement BEFORE running the function
    assert function_globals["openai"] is fake_client

    brochure, status = generate_brochure_stream(
        company_name="TEST",
        context="Simulated company information",
        audience="customers"
    )

finally:
    # Always restore the real client
    function_globals["openai"] = original_client

# Confirm the fake API was actually used
assert fake_create.call_count == 1

expected_text = "This is an interrupted "

assert brochure == expected_text
assert status == "partial"

print("\nMock API calls:", fake_create.call_count)
print("Status:", status)
print("Accumulated text:", repr(brochure))
print("Test passed!")

Generating TEST brochure for customers...


This is an interrupted 


Streaming finished.
Status: partial
Run ID: 7e6da9a2-383a-4288-af90-acdeead82896
Duration: 0.02 seconds
Saved to: outputs/streaming/TEST_customers_7e6da9a2-383a-4288-af90-acdeead82896.md
Metadata saved to: outputs/streaming/7e6da9a2-383a-4288-af90-acdeead82896_metadata.json

Mock API calls: 1
Status: partial
Accumulated text: 'This is an interrupted '
Test passed!


In [163]:
from pathlib import Path
import json

run_id = "7e6da9a2-383a-4288-af90-acdeead82896"

brochure_path = Path(
    f"outputs/streaming/TEST_customers_{run_id}.md"
)

metadata_path = Path(
    f"outputs/streaming/{run_id}_metadata.json"
)

saved_text = brochure_path.read_text(encoding="utf-8")
metadata = json.loads(metadata_path.read_text(encoding="utf-8"))

expected_text = "This is an interrupted "

assert saved_text == expected_text
assert metadata["status"] == "partial"
assert metadata["characters"] == len(expected_text)
assert metadata["error"] == "Simulated connection failure"
assert Path(metadata["output_path"]).resolve() == brochure_path.resolve()
assert "None" not in saved_text

print("Saved text:", repr(saved_text))
print("Saved status:", metadata["status"])
print("Saved error:", metadata["error"])
print("Saved characters:", metadata["characters"])
print("\nAll saved-stream verification tests passed!")

Saved text: 'This is an interrupted '
Saved status: partial
Saved error: Simulated connection failure
Saved characters: 23

All saved-stream verification tests passed!


In [164]:
# save streaning test results
from pathlib import Path
import json

tests_dir = Path("tests")
tests_dir.mkdir(parents=True, exist_ok=True)

# Files created during our streaming tests
completed_path = tests_dir / "streaming_simulated_output.txt"
interrupted_path = tests_dir / "streaming_interrupted_output.txt"

# Metadata from the full mocked streaming workflow
run_id = "7e6da9a2-383a-4288-af90-acdeead82896"
metadata_path = Path(f"outputs/streaming/{run_id}_metadata.json")

metadata = json.loads(metadata_path.read_text(encoding="utf-8"))

# Read actual saved outputs
completed_text = completed_path.read_text(encoding="utf-8")
interrupted_text = interrupted_path.read_text(encoding="utf-8")
workflow_text = Path(metadata["output_path"]).read_text(encoding="utf-8")

# Verify the saved outputs
checks = {
    "completed_text_exact": completed_text == "Hello world!",
    "interrupted_text_exact": interrupted_text == "This is an interrupted ",
    "workflow_text_exact": workflow_text == "This is an interrupted ",
    "interruption_marked_partial": metadata["status"] == "partial",
    "interruption_error_recorded": metadata["error"] == "Simulated connection failure",
    "character_count_correct": metadata["characters"] == len(workflow_text),
    "no_none_in_outputs": all(
        "None" not in text
        for text in [completed_text, interrupted_text, workflow_text]
    )
}

assert all(checks.values()), checks

report = {
    "test_type": "streaming_simulation_and_interruption",
    "status": "PASS",
    "api_calls": 0,
    "tests": {
        "normal_stream": {
            "expected_status": "completed",
            "expected_text": "Hello world!",
            "output_path": str(completed_path)
        },
        "interrupted_stream": {
            "expected_status": "partial",
            "expected_text": "This is an interrupted ",
            "output_path": str(interrupted_path)
        },
        "full_workflow_mock": {
            "run_id": run_id,
            "status": metadata["status"],
            "output_path": metadata["output_path"],
            "metadata_path": str(metadata_path),
            "error": metadata["error"]
        }
    },
    "checks": checks
}

report_path = tests_dir / "streaming_test_results.json"

report_path.write_text(
    json.dumps(report, indent=4, ensure_ascii=False),
    encoding="utf-8"
)

print("All streaming checks passed!")
print("Results saved to:", report_path)

All streaming checks passed!
Results saved to: tests\streaming_test_results.json


## Audit logging and reproducibility

In [166]:
from pathlib import Path
from datetime import datetime, timezone
from uuid import uuid4
import json

RUN_LOG_PATH = Path("logs/run_log.jsonl")

def log_operation(
    operation,
    company=None,
    audience=None,
    model=None,
    duration_seconds=None,
    status="success",
    input_paths=None,
    output_paths=None,
    error=None,
    token_usage=None,
    run_id=None,
    api_calls=0,
    http_requests=0
):
    """
    Append one execution record to the central run log.

    This function does not call any API.
    """

    record = {
        "run_id": run_id or str(uuid4()),
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "operation": operation,
        "company": company,
        "audience": audience,
        "model": model,
        "duration_seconds": duration_seconds,
        "status": status,
        "input_paths": input_paths or [],
        "output_paths": output_paths or [],
        "error": error,
        "token_usage": token_usage,
        "api_calls": api_calls,
        "http_requests": http_requests
    }

    RUN_LOG_PATH.parent.mkdir(parents=True, exist_ok=True)

    with RUN_LOG_PATH.open("a", encoding="utf-8") as file:
        file.write(json.dumps(record, ensure_ascii=False) + "\n")

    return record

print("Logging function defined successfully.")

Logging function defined successfully.


In [169]:
from pathlib import Path
import json

log_path = Path("logs/run_log.jsonl")

# Read existing records, if any
existing_records = []

if log_path.exists():
    with log_path.open("r", encoding="utf-8") as file:
        existing_records = [
            json.loads(line)
            for line in file
            if line.strip()
        ]

existing_ids = {
    record["run_id"]
    for record in existing_records
}

imported = 0

metadata_files = (
    list(Path("outputs/streaming").glob("*_metadata.json"))
    + list(Path("outputs/comparisons").glob("*_metadata.json"))
)

for metadata_path in sorted(metadata_files):

    metadata = json.loads(
        metadata_path.read_text(encoding="utf-8")
    )

    run_id = metadata["run_id"]

    if run_id in existing_ids:
        continue

    is_streaming = metadata_path.parent.name == "streaming"
    is_mock = run_id == "7e6da9a2-383a-4288-af90-acdeead82896"

    record = {
        "run_id": run_id,
        "timestamp": metadata.get("timestamp"),
        "operation": (
            "streaming_brochure_generation"
            if is_streaming
            else "company_comparison"
        ),
        "company": metadata.get(
            "company",
            ", ".join(metadata.get("companies", []))
        ),
        "audience": metadata.get("audience"),
        "model": "mock-model" if is_mock else metadata.get("model"),
        "duration_seconds": metadata.get("duration_seconds"),
        "status": metadata.get("status"),
        "input_paths": (
            [metadata["prompt_path"]]
            if metadata.get("prompt_path")
            else []
        ),
        "output_paths": [
            path for path in [
                metadata.get("output_path"),
                str(metadata_path)
            ]
            if path
        ],
        "error": metadata.get("error"),
        "token_usage": None,
        "api_calls": 0 if is_mock else 1,
        "http_requests": 0,
        "execution_type": "mock_test" if is_mock else "real_api",
        "historical_import": True
    }

    log_path.parent.mkdir(parents=True, exist_ok=True)

    with log_path.open("a", encoding="utf-8") as file:
        file.write(
            json.dumps(record, ensure_ascii=False) + "\n"
        )

    existing_ids.add(run_id)
    imported += 1

print("Historical records imported:", imported)
print("Total records:", len(existing_ids))
print("Log file:", log_path)

Historical records imported: 4
Total records: 4
Log file: logs\run_log.jsonl


In [172]:

from pathlib import Path
import json

log_path = Path("logs/run_log.jsonl")

# Existing run IDs, to avoid duplicate entries
existing_ids = set()

if log_path.exists():
    with log_path.open("r", encoding="utf-8") as file:
        for line in file:
            if line.strip():
                existing_ids.add(json.loads(line)["run_id"])

# Historical operations to recover
historical_runs = [
    {
        "run_id": "b9839cc2-660b-4f76-ba9f-48aeff5992ba",
        "operation": "brochure_generation",
        "company": "NVIDIA",
        "audience": "candidates",
        "output_path": "outputs/brochures/A_candidates_b9839cc2-660b-4f76-ba9f-48aeff5992ba.md"
    },
    {
        "run_id": "19b70913-5646-491e-9710-851aad2887f3",
        "operation": "brochure_generation",
        "company": "AMD",
        "audience": "customers",
        "output_path": "outputs/brochures/B_customers_19b70913-5646-491e-9710-851aad2887f3.md"
    },
    {
        "run_id": "a632ccd7-3488-4b44-acee-bf018dd4a401",
        "operation": "brochure_generation",
        "company": "AMD",
        "audience": "candidates",
        "output_path": "outputs/brochures/B_candidates_a632ccd7-3488-4b44-acee-bf018dd4a401.md"
    },
    {
        "run_id": "d6149c02-7f96-4d30-b5f6-f14663fe2e71",
        "operation": "baseline_experiment",
        "company": "NVIDIA",
        "audience": "customers",
        "output_path": "outputs/baseline/NVIDIA_customers_baseline.md",
        "prompt_path": "data/prompts/NVIDIA_customers_baseline.txt",
        "duration_seconds": 7.58
    }
]

imported = 0

for run in historical_runs:
    if run["run_id"] in existing_ids:
        print(
            "ALREADY LOGGED:",
            run["company"],
            run["audience"]
        )
        continue

    output_path = Path(run["output_path"])

    if not output_path.is_file():
        print("SKIPPED — missing output:", output_path)
        continue

    input_paths = []

    if run.get("prompt_path"):
        prompt_path = Path(run["prompt_path"])

        if prompt_path.is_file():
            input_paths.append(str(prompt_path))

    record = {
        "run_id": run["run_id"],
        "timestamp": None,
        "operation": run["operation"],
        "company": run["company"],
        "audience": run["audience"],
        "model": globals().get("MODEL"),
        "duration_seconds": run.get("duration_seconds"),
        "status": "success",
        "input_paths": input_paths,
        "output_paths": [str(output_path)],
        "error": None,
        "token_usage": None,
        "api_calls": None,
        "http_requests": 0,
        "execution_type": "historical_real_generation",
        "historical_import": True,
        "metadata_note": (
            "Recovered from existing output. "
            "Missing execution details were not estimated."
        )
    }

    log_path.parent.mkdir(parents=True, exist_ok=True)

    with log_path.open("a", encoding="utf-8") as file:
        file.write(
            json.dumps(record, ensure_ascii=False) + "\n"
        )

    existing_ids.add(run["run_id"])
    imported += 1

    print(
        f"IMPORTED: {run['company']} "
        f"{run['audience']} — {run['operation']}"
    )

print("\nNew records imported:", imported)
print("Total records:", len(existing_ids))
print("Log file:", log_path)


IMPORTED: NVIDIA candidates — brochure_generation
IMPORTED: AMD customers — brochure_generation
IMPORTED: AMD candidates — brochure_generation
IMPORTED: NVIDIA customers — baseline_experiment

New records imported: 4
Total records: 8
Log file: logs\run_log.jsonl


In [174]:
# update - responsible for: generating, saving, and logging the brochure.
def generate_brochure(company_name, context, audience, company_id=None):
    """
    Generate, save, and log a company brochure.

    Returns the generated brochure text.
    """

    import time
    from datetime import datetime, timezone
    from uuid import uuid4
    from pathlib import Path

    run_id = str(uuid4())
    start_time = time.perf_counter()

    print(f"Generating {company_name} brochure for {audience}...")

    output_path = None
    prompt_path = None
    api_calls = 0
    token_usage = None
    status = "failure"
    error_message = None

    # Use company ID if supplied, otherwise use company name
    file_prefix = company_id or company_name

    try:
        # Build prompt
        prompt = build_brochure_prompt(
            company_name,
            audience,
            context
        )

        # Save exact prompt for reproducibility
        prompt_dir = Path("data/prompts")
        prompt_dir.mkdir(parents=True, exist_ok=True)

        prompt_path = prompt_dir / f"{file_prefix}_{audience}_{run_id}.txt"
        prompt_path.write_text(prompt, encoding="utf-8")

        # Call Gemini through the OpenAI-compatible API
        # Disable automatic SDK retries so each attempt is explicit.
        api_calls += 1

        response = openai.with_options(
            max_retries=0
        ).chat.completions.create(
            model=MODEL,
            messages=[
                {
                    "role": "user",
                    "content": prompt
                }
            ]
        )

        brochure = response.choices[0].message.content

        if not isinstance(brochure, str) or not brochure.strip():
            raise ValueError("Gemini returned empty brochure content")

        # Capture token usage only when actually reported
        usage = getattr(response, "usage", None)

        if usage is not None:
            token_usage = (
                usage.model_dump(exclude_none=True)
                if hasattr(usage, "model_dump")
                else None
            )

        # Save brochure with the SAME run ID
        output_dir = Path("outputs/brochures")
        output_dir.mkdir(parents=True, exist_ok=True)

        output_path = output_dir / (
            f"{file_prefix}_{audience}_{run_id}.md"
        )

        output_path.write_text(
            brochure,
            encoding="utf-8"
        )

        status = "success"

        return brochure

    except Exception as error:
        error_message = str(error)
        raise

    finally:
        duration = round(time.perf_counter() - start_time, 2)

        log_operation(
            run_id=run_id,
            operation="brochure_generation",
            company=company_name,
            audience=audience,
            model=MODEL,
            duration_seconds=duration,
            status=status,
            input_paths=(
                [str(prompt_path)]
                if prompt_path is not None
                else []
            ),
            output_paths=(
                [str(output_path)]
                if output_path is not None and output_path.is_file()
                else []
            ),
            error=error_message,
            token_usage=token_usage,
            api_calls=api_calls,
            http_requests=0
        )


Prepare automatic logging for link selection and website fetching

In [181]:
# updated func
def select_relevant_links(url):
    """
    Select company links using Gemini, validate the complete
    response, and automatically save execution logs.
    """

    import time
    from pathlib import Path
    from uuid import uuid4

    run_id = str(uuid4())
    start_time = time.perf_counter()

    api_calls = 0
    http_requests = 0
    token_usage = None
    status = "failure"
    error_message = None

    prompt_path = None
    response_path = None
    candidate_path = None

    try:
        # This function fetches the homepage links once.
        # We count the attempted website request separately.
        http_requests += 1

        user_prompt = get_links_user_prompt(url)
        candidate_links = list(normalized_candidate_links)

        # Save exact inputs
        prompt_dir = Path("data/prompts")
        prompt_dir.mkdir(parents=True, exist_ok=True)

        prompt_path = prompt_dir / f"link_selection_{run_id}.json"

        prompt_data = {
            "run_id": run_id,
            "company_url": url,
            "model": MODEL,
            "system_prompt": link_system_prompt,
            "user_prompt": user_prompt,
            "candidate_links": candidate_links
        }

        prompt_path.write_text(
            json.dumps(prompt_data, indent=4, ensure_ascii=False),
            encoding="utf-8"
        )

        # Gemini request, with automatic retries disabled
        api_calls += 1

        response = openai.with_options(
            max_retries=0
        ).chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": link_system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            response_format={"type": "json_object"}
        )

        usage = getattr(response, "usage", None)

        if usage is not None and hasattr(usage, "model_dump"):
            token_usage = usage.model_dump(exclude_none=True)

        raw_text = response.choices[0].message.content

        # Save raw response, even if validation fails
        response_dir = Path("data/responses")
        response_dir.mkdir(parents=True, exist_ok=True)

        response_path = response_dir / f"link_selection_{run_id}.txt"

        response_path.write_text(
            raw_text if isinstance(raw_text, str) else "",
            encoding="utf-8"
        )

        # Existing strict validation
        selected = validate_link_selection(
            raw_text,
            url,
            candidate_links
        )

        status = "success"
        print("Gemini selection validated successfully.")

        return selected

    except Exception as error:
        error_message = str(error)

        # Preserve existing rejection evidence
        if isinstance(error, ValueError):
            rejection_dir = Path("logs/link_rejections")
            rejection_dir.mkdir(parents=True, exist_ok=True)

            rejection_path = rejection_dir / f"link_selection_{run_id}.json"
            rejection_path.parent.mkdir(parents=True, exist_ok=True)

            rejection = {
                "run_id": run_id,
                "company_url": url,
                "raw_response": (
                    raw_text if "raw_text" in locals() else None
                ),
                "candidate_links": (
                    candidate_links if "candidate_links" in locals() else []
                ),
                "rejection_reason": error_message
            }

            rejection_path.write_text(
                json.dumps(rejection, indent=4, ensure_ascii=False),
                encoding="utf-8"
            )

        raise

    finally:
        duration = round(time.perf_counter() - start_time, 2)

        log_operation(
            run_id=run_id,
            operation="link_selection",
            company=url,
            model=MODEL,
            duration_seconds=duration,
            status=status,
            input_paths=[
                str(path)
                for path in [prompt_path]
                if path is not None and path.is_file()
            ],
            output_paths=[
                str(path)
                for path in [response_path]
                if path is not None and path.is_file()
            ],
            error=error_message,
            token_usage=token_usage,
            api_calls=api_calls,
            http_requests=http_requests
        )


In [183]:
# updated func

def fetch_company_sources(company_id, url, selected):
    """
    Fetch the homepage and selected pages, save source records,
    and log HTTP request attempts and execution results.
    """

    import time
    from pathlib import Path
    from uuid import uuid4

    run_id = str(uuid4())
    start_time = time.perf_counter()

    source_prefix = company_id.upper()

    pages = [{"url": url, "type": "homepage"}]

    for link in selected["links"]:
        pages.append({
            "url": link["url"],
            "type": link["type"]
        })

    sources = []
    http_requests = 0
    error_messages = []
    output_path = None

    try:
        for index, page_info in enumerate(pages, start=1):
            source_id = f"{source_prefix}{index}"
            page_url = page_info["url"]

            print(f"Fetching {source_id}: {page_url}")

            source = {
                "source_id": source_id,
                "company_id": source_prefix,
                "url": page_url,
                "title": "",
                "retrieved_at": datetime.now(timezone.utc).isoformat(),
                "text": "",
                "status": "failed"
            }

            try:
                # One attempted HTTP request per page.
                http_requests += 1

                content = fetch_website_contents(page_url)

                if content:
                    title, separator, text = content.partition("\n\n")

                    source["title"] = title.strip()
                    source["text"] = text.strip() if separator else ""

                    source["status"] = (
                        "ok" if source["text"] else "empty"
                    )
                else:
                    source["status"] = "empty"

            except Exception as error:
                error_messages.append(
                    f"{source_id}: {error}"
                )
                print(f"Failed to fetch {source_id}: {error}")

            sources.append(source)

        output_dir = Path("data/sources")
        output_dir.mkdir(parents=True, exist_ok=True)

        output_path = output_dir / f"{source_prefix}_sources.json"

        output_path.write_text(
            json.dumps(sources, indent=4, ensure_ascii=False),
            encoding="utf-8"
        )

        print(f"\nSaved {len(sources)} sources to {output_path}")

        return sources

    finally:
        duration = round(time.perf_counter() - start_time, 2)

        ok_count = sum(
            source["status"] == "ok" for source in sources
        )

        if not sources or ok_count == 0:
            status = "failure"
        elif error_messages or ok_count < len(sources):
            status = "partial"
        else:
            status = "success"

        log_operation(
            run_id=run_id,
            operation="source_fetching",
            company=source_prefix,
            model=None,
            duration_seconds=duration,
            status=status,
            input_paths=[],
            output_paths=(
                [str(output_path)]
                if output_path is not None and output_path.is_file()
                else []
            ),
            error=(
                "; ".join(error_messages)
                if error_messages else None
            ),
            token_usage=None,
            api_calls=0,
            http_requests=http_requests
        )


In [185]:
# updated func

def generate_brochure_stream(company_name, context, audience):
    import time
    from pathlib import Path

    run_id = str(uuid4())
    start_time = time.perf_counter()

    brochure = ""
    status = "failure"
    error_message = None
    token_usage = None
    api_calls = 0

    prompt_path = None
    output_path = None
    metadata_path = None

    print(f"Generating {company_name} brochure for {audience}...")

    try:
        prompt = build_brochure_prompt(
            company_name, audience, context
        )

        # Save the exact prompt before making the API call
        prompt_dir = Path("data/prompts")
        prompt_dir.mkdir(parents=True, exist_ok=True)

        prompt_path = prompt_dir / f"streaming_{run_id}.txt"
        prompt_path.write_text(prompt, encoding="utf-8")

        display_handle = display(
            Markdown(""),
            display_id=True
        )

        def update_display(text):
            nonlocal brochure
            brochure = text
            display_handle.update(Markdown(text))

        # Automatic SDK retries disabled
        api_calls += 1

        stream = openai.with_options(
            max_retries=0
        ).chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "user", "content": prompt}
            ],
            stream=True,
            stream_options={"include_usage": True}
        )

        # Collect text and optional usage metadata
        def tracked_stream():
            nonlocal token_usage

            for event in stream:
                usage = getattr(event, "usage", None)

                if usage is not None:
                    if hasattr(usage, "model_dump"):
                        token_usage = usage.model_dump(
                            exclude_none=True
                        )

                yield event

        result = process_stream_events(
            tracked_stream(),
            on_update=update_display
        )

        brochure = result["text"]
        status = result["status"]
        error_message = result.get("error")

    except Exception as error:
        error_message = str(error)

        # Preserve partial text already received
        status = "partial" if brochure else "failure"

    finally:
        duration = round(time.perf_counter() - start_time, 2)

        output_dir = Path("outputs/streaming")
        output_dir.mkdir(parents=True, exist_ok=True)

        output_path = output_dir / (
            f"{company_name}_{audience}_{run_id}.md"
        )

        output_path.write_text(
            brochure,
            encoding="utf-8"
        )

        metadata = {
            "run_id": run_id,
            "company": company_name,
            "audience": audience,
            "model": MODEL,
            "timestamp": datetime.now(timezone.utc).isoformat(),
            "duration_seconds": duration,
            "status": status,
            "output_path": str(output_path),
            "characters": len(brochure),
            "error": error_message,
            "token_usage": token_usage
        }

        metadata_path = output_dir / f"{run_id}_metadata.json"

        metadata_path.write_text(
            json.dumps(
                metadata,
                indent=4,
                ensure_ascii=False
            ),
            encoding="utf-8"
        )

        log_operation(
            run_id=run_id,
            operation="streaming_brochure_generation",
            company=company_name,
            audience=audience,
            model=MODEL,
            duration_seconds=duration,
            status=status,
            input_paths=(
                [str(prompt_path)]
                if prompt_path is not None and prompt_path.is_file()
                else []
            ),
            output_paths=[
                str(output_path),
                str(metadata_path)
            ],
            error=error_message,
            token_usage=token_usage,
            api_calls=api_calls,
            http_requests=0
        )

    print("\nStreaming finished.")
    print("Status:", status)
    print("Run ID:", run_id)
    print("Duration:", duration, "seconds")
    print("Saved to:", output_path)
    print("Metadata saved to:", metadata_path)

    return brochure, status


In [187]:
# updated func

def generate_comparison(context):
    import time
    from pathlib import Path
    from uuid import uuid4
    from datetime import datetime, timezone
    import json

    run_id = str(uuid4())
    start_time = time.perf_counter()

    prompt_path = None
    output_path = None
    metadata_path = None
    token_usage = None
    api_calls = 0
    status = "failure"
    error_message = None

    print("Generating NVIDIA vs AMD comparison...")

    try:
        prompt = build_comparison_prompt(context)

        prompt_dir = Path("data/prompts")
        prompt_dir.mkdir(parents=True, exist_ok=True)

        prompt_path = prompt_dir / f"comparison_{run_id}.txt"
        prompt_path.write_text(prompt, encoding="utf-8")

        api_calls += 1

        response = openai.with_options(
            max_retries=0
        ).chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "user", "content": prompt}
            ]
        )

        usage = getattr(response, "usage", None)
        if usage is not None and hasattr(usage, "model_dump"):
            token_usage = usage.model_dump(exclude_none=True)

        if not response.choices:
            raise ValueError("Model returned no choices")

        comparison = response.choices[0].message.content

        if not isinstance(comparison, str) or not comparison.strip():
            raise ValueError("Model returned empty content")

        output_dir = Path("outputs/comparisons")
        output_dir.mkdir(parents=True, exist_ok=True)

        output_path = output_dir / f"NVIDIA_AMD_{run_id}.md"
        output_path.write_text(comparison, encoding="utf-8")

        status = "success"

        return comparison, {
            "run_id": run_id,
            "status": status,
            "output_path": str(output_path)
        }

    except Exception as error:
        error_message = str(error)
        raise

    finally:
        duration = round(time.perf_counter() - start_time, 2)

        output_dir = Path("outputs/comparisons")
        output_dir.mkdir(parents=True, exist_ok=True)

        metadata_path = output_dir / f"{run_id}_metadata.json"

        metadata = {
            "run_id": run_id,
            "action": "company_comparison",
            "companies": ["NVIDIA", "AMD"],
            "model": MODEL,
            "timestamp": datetime.now(timezone.utc).isoformat(),
            "duration_seconds": duration,
            "status": status,
            "prompt_path": str(prompt_path) if prompt_path else None,
            "output_path": str(output_path) if output_path else None,
            "error": error_message,
            "token_usage": token_usage
        }

        metadata_path.write_text(
            json.dumps(metadata, indent=4, ensure_ascii=False),
            encoding="utf-8"
        )

        log_operation(
            run_id=run_id,
            operation="company_comparison",
            company="NVIDIA, AMD",
            model=MODEL,
            duration_seconds=duration,
            status=status,
            input_paths=(
                [str(prompt_path)]
                if prompt_path is not None and prompt_path.is_file()
                else []
            ),
            output_paths=[
                str(path)
                for path in [output_path, metadata_path]
                if path is not None and path.is_file()
            ],
            error=error_message,
            token_usage=token_usage,
            api_calls=api_calls,
            http_requests=0
        )


In [188]:
#updated func

def generate_baseline(company_name, company_id, audience, context):
    """
    Generate, save, and log a baseline brochure.
    Uses the same model settings as the improved version.
    """

    import time
    import json
    from pathlib import Path
    from uuid import uuid4

    run_id = str(uuid4())
    start_time = time.perf_counter()

    prompt_path = None
    output_path = None
    token_usage = None
    api_calls = 0
    status = "failure"
    error_message = None

    print(f"Generating {company_name} {audience} baseline...")

    try:
        # Only the prompt wording differs from the improved version
        prompt = build_baseline_prompt(
            company_name, audience, context
        )

        prompt_dir = Path("data/prompts")
        prompt_dir.mkdir(parents=True, exist_ok=True)

        prompt_path = prompt_dir / (
            f"{company_id}_{audience}_baseline_{run_id}.txt"
        )
        prompt_path.write_text(prompt, encoding="utf-8")

        api_calls += 1

        response = openai.with_options(
            max_retries=0
        ).chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "user", "content": prompt}
            ]
        )

        usage = getattr(response, "usage", None)
        if usage is not None and hasattr(usage, "model_dump"):
            token_usage = usage.model_dump(exclude_none=True)

        if not response.choices:
            raise ValueError("Model returned no choices")

        brochure = response.choices[0].message.content

        if not isinstance(brochure, str) or not brochure.strip():
            raise ValueError("Model returned empty baseline content")

        output_dir = Path("outputs/baseline")
        output_dir.mkdir(parents=True, exist_ok=True)

        output_path = output_dir / (
            f"{company_id}_{audience}_baseline_{run_id}.md"
        )
        output_path.write_text(brochure, encoding="utf-8")

        status = "success"
        return brochure

    except Exception as error:
        error_message = str(error)
        raise

    finally:
        duration = round(time.perf_counter() - start_time, 2)

        log_operation(
            run_id=run_id,
            operation="baseline_experiment",
            company=company_name,
            audience=audience,
            model=MODEL,
            duration_seconds=duration,
            status=status,
            input_paths=(
                [str(prompt_path)]
                if prompt_path is not None and prompt_path.is_file()
                else []
            ),
            output_paths=(
                [str(output_path)]
                if output_path is not None and output_path.is_file()
                else []
            ),
            error=error_message,
            token_usage=token_usage,
            api_calls=api_calls,
            http_requests=0
        )

